# PRE-01 — CICIDS2017 Partitioning and Leakage Control


These notebooks implement **Experiment 1 v2 Sections 8–11 only**.

They consume the validated/frozen outputs of Sections 1–7 and construct immutable
partition membership at the **feature-fingerprint-group** level.

They do **not** repeat the Sections 1–7 audits, fit preprocessing, learn categorical
vocabularies, scale/impute/encode features, train AE/DAE models, compute anomaly
scores, or fit anomaly thresholds.

The stage boundary is:

```text
Sections 1–7 validated model-visible corpus
        ↓
PRE-01 Sections 8–11
immutable leakage-controlled membership
        ↓
Notebook 00
FIT learned preprocessing on benign_weight_train only
TRANSFORM all five frozen experimental partitions, including final_test
        ↓
Notebooks 01–06: development only
        ↓
Notebook 07: first authorized final-test evaluation
```


# Scope and prohibited operations

## Scientific role

This notebook answers one question:

> Which immutable observations/fingerprint groups belong to each experimental role?

The five experimental partitions are:

- `benign_weight_train`
- `benign_early_stop`
- `benign_threshold_calibration`
- `development`
- `final_test`

An optional governance-only state, `attack_reserve_unused`, may contain unavoidable
excess **pure-attack fingerprint groups**.

## Hard prohibitions

This notebook must never:

- split a `feature_fingerprint` group;
- create a benign reserve;
- put attacks in benign-only roles;
- put mixed-label groups in benign-only roles or reserve;
- fit or instantiate learned preprocessing;
- infer encoder vocabularies or imputation/scaler parameters;
- train a model;
- compute reconstruction/anomaly scores or thresholds;
- use final-test outcomes for model-development decisions.

**Final-test transformation later is permitted; final-test influence is forbidden.**


In [1]:
# ============================================================
# 3. EDITABLE CONFIGURATION
# ============================================================
from pathlib import Path

DATASET_NAME = "CICIDS2017"
DATASET_PREFIX = "CICIDS2017"
PROTOCOL_VERSION = "Experiment_1_v2"
NOTEBOOK_STAGE = "PRE_01_SECTIONS_8_TO_11"

# Do NOT create /content/drive/MyDrive/... here.
# The next cell verifies a real Drive mount and project root first.

PROJECT_FOLDER_NAME = "EXPERIMENT1_V2"
DATASET_RELATIVE_PATH = Path("DATASETS") / "CICIDS2017"

PRIMARY_DRIVE_MOUNTPOINT = Path("/content/drive")
FALLBACK_DRIVE_MOUNTPOINT_BASE = Path("/content/exp1v2_google_drive")

# Optional escape hatches. Normally leave None.
PROJECT_ROOT_OVERRIDE = None
CICIDS_PREPARTITION_ROOT = None
MASTER_DATASET_PATH = None
GLOBAL_PROTOCOL_PATH_OVERRIDE = None

# Bound only after verified Drive/project discovery.
DRIVE_SEARCH_ROOT = None
DRIVE_MYDRIVE_PATH = None
PROJECT_ROOT = None
DATASET_ROOT = None
UPSTREAM_DIR = None
GLOBAL_PROTOCOL_PATH = None
OUTPUT_DIR = None

MASTER_MANIFEST_PATH = None
TAXONOMY_MANIFEST_PATH = None
ROW_IDENTITY_MANIFEST_PATH = None
RAW_FEATURE_POLICY_PATH = None
FINGERPRINT_AUDIT_PATH = None
FINGERPRINT_ROW_MAPPING_PATH = None
PREPARTITION_READINESS_REPORT_PATH = None
DUPLICATE_GROUP_MANIFEST_PATH = None
CONFLICTING_FINGERPRINT_MANIFEST_PATH = None

# Prospectively frozen for Experiment 1 v2 partition construction.
PARTITION_SEED = 20260914
MAX_MIXED_ATTACK_PREVALENCE = 0.49
OUTPUT_SUBDIR_NAME = "partitioning_s8_s11_v1_seed20260914"

DATASET_LABEL_COLUMNS = ["Label", "binary_label"]
DATASET_ATTACK_FAMILY_COLUMNS = ["Label", "attack_family"]

print("Dataset:", DATASET_NAME)
print("PARTITION_SEED:", PARTITION_SEED)
print("Drive/project paths will be resolved after verified mount.")


Dataset: CICIDS2017
PARTITION_SEED: 20260914
Drive/project paths will be resolved after verified mount.


In [2]:
# ============================================================
# 4. VERIFIED GOOGLE DRIVE MOUNT / PROJECT PATH BINDING
# ============================================================
# A directory named /content/drive/MyDrive is NOT sufficient evidence that
# Google Drive is mounted. Accept a mount only when the real EXPERIMENT1_V2
# project is visible.

from pathlib import Path
import os

def _find_mydrive(mountpoint):
    mountpoint = Path(mountpoint)
    for name in ["MyDrive", "My Drive"]:
        p = mountpoint / name
        if p.exists() and p.is_dir():
            return p
    return None

def _find_project(mydrive, project_name):
    if mydrive is None:
        return None

    direct = Path(mydrive) / project_name
    if direct.exists() and direct.is_dir():
        return direct.resolve()

    matches = []
    for dirpath, dirnames, _ in os.walk(
        str(mydrive),
        topdown=True,
        followlinks=True,
    ):
        dirnames[:] = [
            d for d in dirnames
            if d not in {".Trash", ".trash", "__pycache__", ".ipynb_checkpoints"}
        ]
        if project_name in dirnames:
            matches.append((Path(dirpath) / project_name).resolve())

    unique = []
    seen = set()
    for p in matches:
        if str(p) not in seen:
            unique.append(p)
            seen.add(str(p))

    if len(unique) == 1:
        return unique[0]
    if len(unique) > 1:
        raise RuntimeError(
            f"Multiple {project_name!r} project directories were found: "
            f"{[str(p) for p in unique]}. Set PROJECT_ROOT_OVERRIDE explicitly."
        )
    return None

def _fresh_mountpoint(base):
    base = Path(base)
    candidates = [base] + [Path(f"{base}_{i}") for i in range(1, 20)]
    for candidate in candidates:
        if not candidate.exists():
            return candidate
        if candidate.is_dir() and not any(candidate.iterdir()):
            return candidate
    raise RuntimeError("No fresh dedicated Drive mountpoint is available under /content.")

try:
    from google.colab import drive
    IN_COLAB = True
except ImportError:
    drive = None
    IN_COLAB = False

if PROJECT_ROOT_OVERRIDE is not None:
    PROJECT_ROOT = Path(PROJECT_ROOT_OVERRIDE).expanduser().resolve()
    if not PROJECT_ROOT.exists() or not PROJECT_ROOT.is_dir():
        raise FileNotFoundError(
            f"PROJECT_ROOT_OVERRIDE does not exist or is not a directory: {PROJECT_ROOT}"
        )
    DRIVE_MYDRIVE_PATH = PROJECT_ROOT.parent
    DRIVE_SEARCH_ROOT = DRIVE_MYDRIVE_PATH.parent

else:
    primary_mydrive = _find_mydrive(PRIMARY_DRIVE_MOUNTPOINT)
    primary_project = _find_project(primary_mydrive, PROJECT_FOLDER_NAME)

    if primary_project is not None:
        DRIVE_MYDRIVE_PATH = primary_mydrive.resolve()
        DRIVE_SEARCH_ROOT = PRIMARY_DRIVE_MOUNTPOINT.resolve()
        PROJECT_ROOT = primary_project.resolve()
        print("Verified existing Drive mount:", DRIVE_SEARCH_ROOT)

    else:
        if not IN_COLAB:
            raise RuntimeError(
                "No verified Drive project is visible and Google Colab is unavailable. "
                "Set PROJECT_ROOT_OVERRIDE explicitly."
            )

        alternate_mountpoint = _fresh_mountpoint(FALLBACK_DRIVE_MOUNTPOINT_BASE)
        print(
            "Conventional /content/drive does not expose the real project. "
            "Mounting Google Drive at:",
            alternate_mountpoint,
        )

        drive.mount(str(alternate_mountpoint), force_remount=False)

        alternate_mydrive = _find_mydrive(alternate_mountpoint)
        if alternate_mydrive is None:
            raise RuntimeError(
                f"Drive mounted at {alternate_mountpoint}, but MyDrive was not exposed."
            )

        alternate_project = _find_project(
            alternate_mydrive,
            PROJECT_FOLDER_NAME,
        )
        if alternate_project is None:
            top_entries = []
            try:
                top_entries = sorted(p.name for p in alternate_mydrive.iterdir())[:50]
            except Exception:
                pass
            raise FileNotFoundError(
                f"Drive mounted successfully at {alternate_mountpoint}, but "
                f"{PROJECT_FOLDER_NAME!r} was not found under {alternate_mydrive}. "
                f"Top-level entries (first 50): {top_entries}. "
                "Set PROJECT_ROOT_OVERRIDE if your project root has another name."
            )

        DRIVE_MYDRIVE_PATH = alternate_mydrive.resolve()
        DRIVE_SEARCH_ROOT = alternate_mountpoint.resolve()
        PROJECT_ROOT = alternate_project.resolve()
        print("Verified project using alternate Drive mount:", PROJECT_ROOT)

DATASET_ROOT = PROJECT_ROOT / DATASET_RELATIVE_PATH
GLOBAL_PROTOCOL_PATH = (
    PROJECT_ROOT / "global" / "experiment1_v2_global_protocol.json"
)
OUTPUT_DIR = DATASET_ROOT / OUTPUT_SUBDIR_NAME

if not DATASET_ROOT.exists():
    raise FileNotFoundError(
        f"Verified project exists but CICIDS2017 dataset root is missing: {DATASET_ROOT}"
    )

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

print("DRIVE_SEARCH_ROOT:", DRIVE_SEARCH_ROOT)
print("DRIVE_MYDRIVE_PATH:", DRIVE_MYDRIVE_PATH)
print("PROJECT_ROOT:", PROJECT_ROOT)
print("DATASET_ROOT:", DATASET_ROOT)
print("OUTPUT_DIR:", OUTPUT_DIR)


Conventional /content/drive does not expose the real project. Mounting Google Drive at: /content/exp1v2_google_drive
Mounted at /content/exp1v2_google_drive
Verified project using alternate Drive mount: /content/exp1v2_google_drive/MyDrive/EXPERIMENT1_V2
DRIVE_SEARCH_ROOT: /content/exp1v2_google_drive
DRIVE_MYDRIVE_PATH: /content/exp1v2_google_drive/MyDrive
PROJECT_ROOT: /content/exp1v2_google_drive/MyDrive/EXPERIMENT1_V2
DATASET_ROOT: /content/exp1v2_google_drive/MyDrive/EXPERIMENT1_V2/DATASETS/CICIDS2017
OUTPUT_DIR: /content/exp1v2_google_drive/MyDrive/EXPERIMENT1_V2/DATASETS/CICIDS2017/partitioning_s8_s11_v1_seed20260914


In [3]:
# ============================================================
# 5. IMPORTS AND ENVIRONMENT CAPTURE
# ============================================================
import gc
import hashlib
import json
import math
import os
import platform
import sys
import tempfile
from collections import Counter, defaultdict
from datetime import datetime, timezone
from pathlib import Path

import numpy as np
import pandas as pd

try:
    import pyarrow as pa
    import pyarrow.parquet as pq
except Exception as exc:
    raise RuntimeError(
        "PyArrow is required for large immutable partition manifests."
    ) from exc

EXECUTION_UTC = datetime.now(timezone.utc).isoformat()

SOFTWARE_ENVIRONMENT = {
    "python_version": sys.version,
    "pandas_version": pd.__version__,
    "numpy_version": np.__version__,
    "pyarrow_version": pa.__version__,
    "platform": platform.platform(),
    "execution_timestamp_utc": EXECUTION_UTC,
}

display(pd.DataFrame(
    [{"component": k, "value": v} for k, v in SOFTWARE_ENVIRONMENT.items()]
))


,component,value
0,python_version,"3.13.16 (main, Oct 1 2026, 15:40:24) [GCC 13...."
1,pandas_version,2.2.3
2,numpy_version,2.1.3
3,pyarrow_version,23.0.1
4,platform,Linux-6.6.122+-x86_64-with-glibc2.39
5,execution_timestamp_utc,2026-10-06T10:38:50.385939+00:00


In [4]:
from pathlib import Path
import json
import hashlib

# Support the two mount styles you have used.
candidate_roots = [
    Path("/content/drive/MyDrive/EXPERIMENT1_V2/DATASETS/CICIDS2017"),
    Path("/content/exp1v2_google_drive/MyDrive/EXPERIMENT1_V2/DATASETS/CICIDS2017"),
    Path("/content/exp1v2_google_drive_1/MyDrive/EXPERIMENT1_V2/DATASETS/CICIDS2017"),
]

root = next((p for p in candidate_roots if p.exists()), None)
assert root is not None, "CICIDS2017 Experiment 1 v2 root not found."

wanted = [
    "fingerprint_audit.json",
    "prepartition_readiness_report.json",
    "row_modelview_group_mapping.parquet",
    "partition_integrity_report.json",
    "partition_membership.parquet",
    "preprocessing_handoff_manifest.json",
]

print("CICIDS2017 ROOT:", root)

for name in wanted:
    hits = sorted(root.rglob(name))
    print(f"\n{name}: {len(hits)} hit(s)")
    for p in hits:
        print("  ", p)

# Also locate files containing the authoritative fingerprint hash.
TARGET_FP_HASH = "1bc46e61bd3c74df45cd21cf280b773693a66898f3a53b744aeb25fd4da00ecf"

print("\nFILES CONTAINING AUTHORITATIVE CICIDS FINGERPRINT HASH")
for p in root.rglob("*.json"):
    try:
        txt = p.read_text(encoding="utf-8")
        if TARGET_FP_HASH in txt:
            print("  ", p)
    except Exception:
        pass

CICIDS2017 ROOT: /content/exp1v2_google_drive/MyDrive/EXPERIMENT1_V2/DATASETS/CICIDS2017

fingerprint_audit.json: 1 hit(s)
   /content/exp1v2_google_drive/MyDrive/EXPERIMENT1_V2/DATASETS/CICIDS2017/CICIDS_Audit_Governance/fingerprints/fingerprint_audit.json

prepartition_readiness_report.json: 1 hit(s)
   /content/exp1v2_google_drive/MyDrive/EXPERIMENT1_V2/DATASETS/CICIDS2017/CICIDS_Audit_Governance/readiness/prepartition_readiness_report.json

row_modelview_group_mapping.parquet: 1 hit(s)
   /content/exp1v2_google_drive/MyDrive/EXPERIMENT1_V2/DATASETS/CICIDS2017/CICIDS_Audit_Governance/fingerprints/row_modelview_group_mapping.parquet

partition_integrity_report.json: 1 hit(s)
   /content/exp1v2_google_drive/MyDrive/EXPERIMENT1_V2/DATASETS/CICIDS2017/partitioning_s8_s11_v1_seed20260914/partition_integrity_report.json

partition_membership.parquet: 1 hit(s)
   /content/exp1v2_google_drive/MyDrive/EXPERIMENT1_V2/DATASETS/CICIDS2017/partitioning_s8_s11_v1_seed20260914/partition_membership

In [ ]:
# ============================================================
# 6. UTILITY / HASH / VALIDATION FUNCTIONS
# ============================================================
BLOCKING_FAILURES = []
NONBLOCKING_WARNINGS = []
ARTIFACT_RECORDS = []

EXPERIMENTAL_PARTITIONS = [
    "benign_weight_train",
    "benign_early_stop",
    "benign_threshold_calibration",
    "development",
    "final_test",
]
ALL_MEMBERSHIP_STATES = EXPERIMENTAL_PARTITIONS + ["attack_reserve_unused"]
BENIGN_ONLY_PARTITIONS = [
    "benign_weight_train",
    "benign_early_stop",
    "benign_threshold_calibration",
]
MIXED_EVALUATION_PARTITIONS = ["development", "final_test"]

def canonical_json_bytes(obj):
    return json.dumps(
        obj,
        sort_keys=True,
        separators=(",", ":"),
        ensure_ascii=False,
        allow_nan=False,
    ).encode("utf-8")

def sha256_bytes(data):
    return hashlib.sha256(data).hexdigest()

def sha256_file(path, block_size=16 * 1024 * 1024):
    path = Path(path)
    h = hashlib.sha256()
    with path.open("rb") as f:
        while True:
            block = f.read(block_size)
            if not block:
                break
            h.update(block)
    return h.hexdigest()

def read_json_required(path, description):
    path = Path(path)
    if not path.exists():
        raise FileNotFoundError(f"Missing required {description}: {path}")
    try:
        with path.open("r", encoding="utf-8") as f:
            return json.load(f)
    except Exception as exc:
        raise RuntimeError(f"Unable to read {description}: {path}") from exc

def record_failure(message):
    BLOCKING_FAILURES.append(str(message))
    print(f"[BLOCKING FAILURE] {message}")

def record_warning(message):
    NONBLOCKING_WARNINGS.append(str(message))
    print(f"[WARNING] {message}")

def require(condition, message):
    if not condition:
        raise RuntimeError(message)

def atomic_write_bytes(path, data):
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)

    with tempfile.NamedTemporaryFile(
        mode="wb", delete=False, dir=str(path.parent), prefix=path.name + ".tmp."
    ) as tmp:
        tmp.write(data)
        tmp.flush()
        os.fsync(tmp.fileno())
        tmp_path = Path(tmp.name)

    new_sha = sha256_file(tmp_path)

    if path.exists():
        old_sha = sha256_file(path)
        if old_sha == new_sha:
            tmp_path.unlink()
            return "REUSED_IDENTICAL", old_sha
        tmp_path.unlink()
        raise RuntimeError(
            f"Refusing to overwrite incompatible frozen artifact: {path}\n"
            f"existing_sha256={old_sha}\nnew_sha256={new_sha}"
        )

    os.replace(tmp_path, path)
    return "WRITTEN", new_sha

def write_json_frozen(path, obj):
    payload = (
        json.dumps(
            obj,
            indent=2,
            sort_keys=True,
            ensure_ascii=False,
            allow_nan=False,
        )
        + "\n"
    ).encode("utf-8")
    return atomic_write_bytes(path, payload)

def write_text_frozen(path, text):
    return atomic_write_bytes(path, str(text).encode("utf-8"))

def write_csv_frozen(path, df):
    return atomic_write_bytes(path, df.to_csv(index=False).encode("utf-8"))

def write_parquet_frozen(path, df, compression="snappy"):
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    tmp_path = path.parent / (path.name + ".tmp.parquet")
    if tmp_path.exists():
        tmp_path.unlink()

    df.to_parquet(
        tmp_path,
        index=False,
        engine="pyarrow",
        compression=compression,
    )
    new_sha = sha256_file(tmp_path)

    if path.exists():
        old_sha = sha256_file(path)
        if old_sha == new_sha:
            tmp_path.unlink()
            return "REUSED_IDENTICAL", old_sha
        tmp_path.unlink()
        raise RuntimeError(
            f"Refusing to overwrite incompatible frozen Parquet artifact: {path}"
        )

    os.replace(tmp_path, path)
    return "WRITTEN", new_sha

def register_artifact(path, purpose, section):
    path = Path(path)
    if not path.exists():
        return
    ARTIFACT_RECORDS.append({
        "artifact_name": path.name,
        "artifact_path": str(path),
        "artifact_type": path.suffix.lower().lstrip(".") or "file",
        "file_size": int(path.stat().st_size),
        "sha256": sha256_file(path),
        "purpose": purpose,
        "section": str(section),
    })

def stable_key64(seed, fingerprint):
    payload = f"{int(seed)}|{fingerprint}".encode("utf-8")
    return int.from_bytes(hashlib.sha256(payload).digest()[:8], "big", signed=False)

def stable_hex(seed, fingerprint):
    return hashlib.sha256(
        f"{int(seed)}|{fingerprint}".encode("utf-8")
    ).hexdigest()

def _unique_resolved_paths(paths):
    out = []
    seen = set()
    for p in paths:
        p = Path(p).resolve()
        key = str(p)
        if key not in seen:
            out.append(p)
            seen.add(key)
    return out

def _safe_json(path):
    try:
        with Path(path).open("r", encoding="utf-8") as f:
            return json.load(f)
    except Exception:
        return None

def walk_find_exact_filenames(roots, target_filenames):
    targets = set(target_filenames)
    found = {name: [] for name in targets}

    for root in [Path(r) for r in roots if r is not None]:
        if not root.exists():
            continue

        for dirpath, dirnames, filenames in os.walk(
            str(root),
            topdown=True,
            followlinks=True,
        ):
            dirnames[:] = [
                d for d in dirnames
                if d not in {".Trash", ".trash", "__pycache__", ".ipynb_checkpoints"}
            ]
            for filename in targets.intersection(filenames):
                found[filename].append(
                    Path(dirpath, filename).resolve()
                )

    for filename in found:
        found[filename] = _unique_resolved_paths(found[filename])
    return found

def candidate_root_from_anchor(path):
    path = Path(path).resolve()
    known_subdirs = {
        "readiness",
        "manifests",
        "taxonomy",
        "identity",
        "feature_policy",
        "fingerprints",
        "audits",
    }
    if path.parent.name in known_subdirs:
        return path.parent.parent
    return path.parent

def collect_required_cicids_pre00_artifacts(root):
    required_names = [
        "prepartition_readiness_report.json",
        "CICIDS2017_master_manifest.json",
        "taxonomy_manifest.json",
        "row_identity_manifest.json",
        "raw_feature_policy.json",
        "fingerprint_audit.json",
        "row_modelview_group_mapping.parquet",
    ]

    hits = walk_find_exact_filenames([root], required_names)
    artifact_map = {}

    for filename in required_names:
        matches = hits.get(filename, [])
        if len(matches) != 1:
            return None
        artifact_map[filename] = matches[0]

    return artifact_map

def validate_cicids_pre00_candidate(root):
    artifact_map = collect_required_cicids_pre00_artifacts(root)
    if artifact_map is None:
        return None

    readiness_path = artifact_map["prepartition_readiness_report.json"]
    report = _safe_json(readiness_path)
    if not isinstance(report, dict):
        return None

    dataset_key = str(report.get("dataset", "")).strip().casefold()
    if dataset_key not in {"cicids2017", "cicids-2017", "cicids_2017"}:
        return None

    if report.get("final_status") not in {
        "PASS",
        "PASS_WITH_NONBLOCKING_WARNINGS",
    }:
        return None

    if any(
        report.get(f"section_{s}_status") != "PASS"
        for s in range(1, 8)
    ):
        return None

    expected_fp_hash = str(
        report.get("fingerprint_manifest_hash", "")
    ).lower()
    if not expected_fp_hash:
        return None

    actual_fp_hash = sha256_file(
        artifact_map["row_modelview_group_mapping.parquet"]
    ).lower()

    if actual_fp_hash != expected_fp_hash:
        return None

    return {
        "root": Path(root).resolve(),
        "readiness_path": readiness_path,
        "readiness": report,
        "artifacts": artifact_map,
    }

def discover_valid_cicids_prepartition_run(
    drive_root,
    explicit_root=None,
):
    if explicit_root is not None:
        explicit_root = Path(explicit_root).resolve()
        if not explicit_root.exists() or not explicit_root.is_dir():
            raise FileNotFoundError(
                f"Explicit CICIDS_PREPARTITION_ROOT does not exist: {explicit_root}"
            )
        validated = validate_cicids_pre00_candidate(explicit_root)
        if validated is None:
            raise RuntimeError(
                "Explicit CICIDS_PREPARTITION_ROOT does not contain one complete, "
                "cryptographically valid CICIDS2017 Sections 1–7 PRE-00 artifact set."
            )
        return validated

    anchor_names = [
        "prepartition_readiness_report.json",
        "row_modelview_group_mapping.parquet",
        "CICIDS2017_master_manifest.json",
        "raw_feature_policy.json",
        "fingerprint_audit.json",
    ]

    anchor_hits = walk_find_exact_filenames([drive_root], anchor_names)
    all_anchor_paths = []
    for filename in anchor_names:
        all_anchor_paths.extend(anchor_hits.get(filename, []))

    candidate_roots = _unique_resolved_paths(
        candidate_root_from_anchor(p)
        for p in all_anchor_paths
    )

    valid = []
    for root in candidate_roots:
        item = validate_cicids_pre00_candidate(root)
        if item is not None:
            valid.append(item)

    dedup = {}
    for item in valid:
        dedup[str(item["root"])] = item
    valid = list(dedup.values())

    if len(valid) == 0:
        diagnostic_counts = {
            filename: len(anchor_hits.get(filename, []))
            for filename in anchor_names
        }
        raise FileNotFoundError(
            "No complete cryptographically valid CICIDS2017 Sections 1–7 PRE-00 "
            f"run was found under verified Drive root {drive_root}. "
            f"Artifact anchor counts={diagnostic_counts}. "
            "Folder names do not matter. If the validated run is outside the "
            "verified Drive root, set CICIDS_PREPARTITION_ROOT explicitly."
        )

    if len(valid) > 1:
        raise RuntimeError(
            "Multiple cryptographically valid CICIDS2017 PRE-00 runs were found. "
            f"Candidates={[str(x['root']) for x in valid]}. "
            "Set CICIDS_PREPARTITION_ROOT explicitly."
        )

    return valid[0]

def resolve_cicids_master_by_hash(
    expected_sha256,
    explicit_path=None,
    preferred_root=None,
    drive_root=None,
):
    expected_sha256 = str(expected_sha256).lower()

    if explicit_path is not None:
        p = Path(explicit_path)
        if not p.exists():
            raise FileNotFoundError(
                f"Configured MASTER_DATASET_PATH is missing: {p}"
            )
        actual = sha256_file(p).lower()
        if actual != expected_sha256:
            raise RuntimeError(
                "Configured MASTER_DATASET_PATH SHA-256 does not match readiness. "
                f"actual={actual}, expected={expected_sha256}"
            )
        return p.resolve()

    known_names = [
        "CICIDS2017_MachineLearningCSV_merged_raw.csv",
        "CICIDS2017_MachineLearningCSV_merged_raw.parquet",
        "CICIDS2017_merged_raw.csv",
        "CICIDS2017_merged_raw.parquet",
    ]

    roots = []
    if preferred_root is not None:
        roots.append(Path(preferred_root))
    if drive_root is not None:
        roots.append(Path(drive_root))

    hits = walk_find_exact_filenames(roots, known_names)
    candidates = []
    for name in known_names:
        candidates.extend(hits.get(name, []))
    candidates = _unique_resolved_paths(candidates)

    matching = [
        p for p in candidates
        if sha256_file(p).lower() == expected_sha256
    ]

    if len(matching) == 1:
        return matching[0]
    if len(matching) > 1:
        raise RuntimeError(
            "Multiple byte-identical CICIDS2017 masters were found. "
            f"Matches={[str(p) for p in matching]}. "
            "Set MASTER_DATASET_PATH explicitly."
        )

    # Renamed master fallback: hash likely CICIDS/master CSV/Parquet files.
    candidates = []
    for root in roots:
        if not root.exists():
            continue
        for dirpath, dirnames, filenames in os.walk(
            str(root),
            topdown=True,
            followlinks=True,
        ):
            dirnames[:] = [
                d for d in dirnames
                if d not in {".Trash", ".trash", "__pycache__", ".ipynb_checkpoints"}
            ]
            for filename in filenames:
                suffix = Path(filename).suffix.lower()
                key = filename.casefold()
                if suffix not in {".csv", ".parquet", ".pq"}:
                    continue
                if (
                    "cicids" in key
                    or "machinelearning" in key
                    or "merged" in key
                ):
                    candidates.append(Path(dirpath, filename).resolve())

    candidates = _unique_resolved_paths(candidates)
    matching = []
    for p in candidates:
        try:
            if sha256_file(p).lower() == expected_sha256:
                matching.append(p)
        except Exception:
            continue

    if len(matching) == 1:
        return matching[0]
    if len(matching) > 1:
        raise RuntimeError(
            "Multiple files match the frozen CICIDS2017 master SHA-256. "
            f"Matches={[str(p) for p in matching]}. "
            "Set MASTER_DATASET_PATH explicitly."
        )

    raise FileNotFoundError(
        "The validated CICIDS2017 readiness report was found, but no CSV/Parquet "
        f"file under the verified Drive root matched frozen master SHA-256 "
        f"{expected_sha256}. Set MASTER_DATASET_PATH explicitly only if the frozen "
        "master is outside the verified Drive tree."
    )

def resolve_global_protocol_by_canonical_hash(
    expected_canonical_hash,
    configured_path=None,
    drive_root=None,
):
    expected = str(expected_canonical_hash).lower()

    def _computed(path):
        obj = read_json_required(path, "global protocol")
        return canonical_hash_from_json_field(
            obj,
            "canonical_protocol_hash",
        ).lower()

    if configured_path is not None:
        p = Path(configured_path)
        if p.exists() and _computed(p) == expected:
            return p.resolve()

    hits = walk_find_exact_filenames(
        [drive_root],
        ["experiment1_v2_global_protocol.json"],
    )["experiment1_v2_global_protocol.json"]

    matching = []
    for p in hits:
        try:
            if _computed(p) == expected:
                matching.append(p)
        except Exception:
            continue

    if len(matching) == 1:
        return matching[0]
    if len(matching) == 0:
        raise FileNotFoundError(
            "No experiment1_v2_global_protocol.json under the verified Drive root "
            f"matched canonical protocol hash {expected}."
        )
    raise RuntimeError(
        "Multiple global protocol files match the same canonical protocol hash. "
        f"Matches={[str(p) for p in matching]}. "
        "Set GLOBAL_PROTOCOL_PATH_OVERRIDE explicitly."
    )

def canonical_hash_from_json_field(obj, stored_hash_key):
    body = dict(obj)
    body.pop(stored_hash_key, None)
    return sha256_bytes(canonical_json_bytes(body))

def largest_remainder(total, weights, tie_order=None):
    """
    Deterministic Hamilton/largest-remainder apportionment.
    Returns integer allocations summing exactly to `total` when total>0 and
    at least one positive weight exists.
    """
    total = int(total)
    keys = list(weights)

    if total < 0:
        raise ValueError("largest_remainder total cannot be negative.")
    if total == 0:
        return {k: 0 for k in keys}

    clean = {k: max(0.0, float(weights[k])) for k in keys}
    denom = sum(clean.values())
    if denom <= 0:
        raise RuntimeError(
            "Largest-remainder allocation requires at least one positive weight."
        )

    quotas = {k: total * clean[k] / denom for k in keys}
    floors = {k: int(math.floor(quotas[k])) for k in keys}
    remaining = total - sum(floors.values())

    if tie_order is None:
        tie_rank = {k: i for i, k in enumerate(sorted(keys, key=str))}
    else:
        tie_rank = {k: i for i, k in enumerate(tie_order)}

    ranked = sorted(
        keys,
        key=lambda k: (
            -(quotas[k] - floors[k]),
            tie_rank.get(k, 10**12),
            str(k),
        ),
    )

    out = dict(floors)
    for k in ranked[:remaining]:
        out[k] += 1

    if sum(out.values()) != total:
        raise RuntimeError("Largest-remainder internal sum mismatch.")
    return out

def parse_row_id_fields(row_ids, dataset_prefix):
    s = pd.Series(row_ids, copy=False, dtype="string")
    prefix = f"{dataset_prefix}::"
    if not bool(s.str.startswith(prefix).all()):
        bad = s[~s.str.startswith(prefix)].head(5).tolist()
        raise RuntimeError(
            f"row_id values do not follow frozen prefix {prefix!r}. Examples={bad}"
        )

    body = s.str[len(prefix):]
    split = body.str.rsplit("::", n=1, expand=True)
    if split.shape[1] != 2:
        raise RuntimeError("Unable to parse source_file/source_row_number from row_id.")

    source_file = split[0].astype("string")
    source_row = pd.to_numeric(split[1], errors="coerce")

    if source_row.isna().any() or (source_row < 1).any():
        raise RuntimeError("row_id contains invalid source-row ordinals.")

    return source_file, source_row.astype("int64")

def normalized_incremental_cost(current, target, add):
    denom = max(1, int(target))
    before = abs(int(current) - int(target)) / denom
    after = abs(int(current) + int(add) - int(target)) / denom
    return after - before

def benign_target_table(B):
    B = int(B)
    targets = {
        "benign_weight_train": int(math.floor(0.50 * B)),
        "benign_early_stop": int(math.floor(0.10 * B)),
        "benign_threshold_calibration": int(math.floor(0.15 * B)),
        "development": int(math.floor(0.10 * B)),
    }
    targets["final_test"] = (
        B
        - targets["benign_weight_train"]
        - targets["benign_early_stop"]
        - targets["benign_threshold_calibration"]
        - targets["development"]
    )
    return targets

def minimum_benign_for_attack_count(attack_count, max_attack_prevalence):
    A = int(attack_count)
    if A <= 0:
        return 0
    # A/(A+B) <= p  =>  B >= A*(1-p)/p
    return int(math.ceil(A * (1.0 - max_attack_prevalence) / max_attack_prevalence))

def attack_capacity_from_benign(benign_count, max_attack_prevalence):
    B = int(benign_count)
    if B <= 0:
        return -1
    r_max = max_attack_prevalence / (1.0 - max_attack_prevalence)
    return int(min(B - 1, math.floor(r_max * B)))

def adjust_attack_targets(
    total_use,
    raw_dev_target,
    dev_cap,
    test_cap,
    mandatory_dev,
    mandatory_test,
):
    total_use = int(total_use)
    dev_cap = int(dev_cap)
    test_cap = int(test_cap)
    mandatory_dev = int(mandatory_dev)
    mandatory_test = int(mandatory_test)

    if mandatory_dev > dev_cap or mandatory_test > test_cap:
        raise RuntimeError(
            "Mandatory mixed-label attack contribution exceeds a hard attack capacity."
        )
    if total_use < mandatory_dev + mandatory_test:
        raise RuntimeError("A_use_target is below mandatory mixed-label attack rows.")
    if total_use > dev_cap + test_cap:
        raise RuntimeError("A_use_target exceeds total hard attack capacity.")

    dev = max(mandatory_dev, min(int(raw_dev_target), dev_cap))
    test = total_use - dev

    if test < mandatory_test:
        shift = mandatory_test - test
        dev -= shift
        test += shift

    if test > test_cap:
        shift = test - test_cap
        test -= shift
        dev += shift

    if dev < mandatory_dev or dev > dev_cap:
        raise RuntimeError("Unable to reconcile development attack target with constraints.")
    if test < mandatory_test or test > test_cap:
        raise RuntimeError("Unable to reconcile final-test attack target with constraints.")
    if dev + test != total_use:
        raise RuntimeError("Adjusted attack targets do not sum to A_use_target.")

    return int(dev), int(test)

def canonical_membership_hash(df):
    required = ["row_id", "partition", "feature_fingerprint"]
    missing = [c for c in required if c not in df.columns]
    if missing:
        raise RuntimeError(f"Canonical membership hash missing columns: {missing}")

    ordered = df[required].sort_values("row_id", kind="mergesort")
    h = hashlib.sha256()
    for row in ordered.itertuples(index=False, name=None):
        for value in row:
            b = str(value).encode("utf-8", errors="surrogatepass")
            h.update(len(b).to_bytes(8, "big"))
            h.update(b)
        h.update(b"\n")
    return h.hexdigest()


# Load and verify Sections 1–7 readiness and frozen identities

This CICIDS2017 PRE-01 notebook no longer assumes a `MERGED` directory.

It first verifies a real Google Drive mount and the actual `EXPERIMENT1_V2`
project root. It then discovers the validated CICIDS2017 Sections 1–7 PRE-00 run
from its frozen artifact set and readiness evidence, irrespective of the PRE-00
folder name.

Finally, it locates the frozen CICIDS2017 master by the SHA-256 recorded in the
validated readiness report. Folder layout is therefore not treated as scientific
identity.


In [ ]:

# ============================================================
# 7–8. LOAD SECTIONS 1–7 READINESS AND VERIFY UPSTREAM HASHES
# ============================================================

PRE00_DISCOVERY = discover_valid_cicids_prepartition_run(
    drive_root=DRIVE_SEARCH_ROOT,
    explicit_root=CICIDS_PREPARTITION_ROOT,
)

UPSTREAM_DIR = PRE00_DISCOVERY["root"]
READINESS = PRE00_DISCOVERY["readiness"]
PREPARTITION_READINESS_REPORT_PATH = PRE00_DISCOVERY["readiness_path"]
artifact_map = PRE00_DISCOVERY["artifacts"]

MASTER_MANIFEST_PATH = artifact_map["CICIDS2017_master_manifest.json"]
TAXONOMY_MANIFEST_PATH = artifact_map["taxonomy_manifest.json"]
ROW_IDENTITY_MANIFEST_PATH = artifact_map["row_identity_manifest.json"]
RAW_FEATURE_POLICY_PATH = artifact_map["raw_feature_policy.json"]
FINGERPRINT_AUDIT_PATH = artifact_map["fingerprint_audit.json"]
FINGERPRINT_ROW_MAPPING_PATH = artifact_map[
    "row_modelview_group_mapping.parquet"
]

# Optional upstream manifests are resolved only inside the same validated PRE-00 root.
optional_hits = walk_find_exact_filenames(
    [UPSTREAM_DIR],
    [
        "duplicate_group_manifest.parquet",
        "conflicting_fingerprint_manifest.parquet",
    ],
)

def optional_unique_from_hits(filename):
    matches = optional_hits.get(filename, [])
    if len(matches) == 0:
        return None
    if len(matches) > 1:
        raise RuntimeError(
            f"Ambiguous optional artifact {filename!r}: {[str(p) for p in matches]}"
        )
    return matches[0]

DUPLICATE_GROUP_MANIFEST_PATH = optional_unique_from_hits(
    "duplicate_group_manifest.parquet"
)
CONFLICTING_FINGERPRINT_MANIFEST_PATH = optional_unique_from_hits(
    "conflicting_fingerprint_manifest.parquet"
)

allowed_upstream_status = {"PASS", "PASS_WITH_NONBLOCKING_WARNINGS"}
if READINESS.get("final_status") not in allowed_upstream_status:
    raise RuntimeError(
        "Partition generation is not authorized because Sections 1–7 readiness is "
        f"{READINESS.get('final_status')!r}."
    )

for s in range(1, 8):
    if READINESS.get(f"section_{s}_status") != "PASS":
        raise RuntimeError(
            f"Upstream Section {s} is not PASS: "
            f"{READINESS.get(f'section_{s}_status')!r}"
        )

EXPECTED_MASTER_SHA256 = str(
    READINESS.get("master_dataset_sha256", "")
).lower()
if not EXPECTED_MASTER_SHA256:
    raise RuntimeError(
        "Validated readiness report does not contain master_dataset_sha256."
    )

MASTER_DATASET_PATH = resolve_cicids_master_by_hash(
    expected_sha256=EXPECTED_MASTER_SHA256,
    explicit_path=MASTER_DATASET_PATH,
    preferred_root=UPSTREAM_DIR,
    drive_root=DRIVE_SEARCH_ROOT,
)

EXPECTED_GLOBAL_PROTOCOL_HASH = str(
    READINESS.get("global_protocol_hash", "")
).lower()
if not EXPECTED_GLOBAL_PROTOCOL_HASH:
    raise RuntimeError(
        "Validated readiness report does not contain global_protocol_hash."
    )

GLOBAL_PROTOCOL_PATH = resolve_global_protocol_by_canonical_hash(
    expected_canonical_hash=EXPECTED_GLOBAL_PROTOCOL_HASH,
    configured_path=(
        GLOBAL_PROTOCOL_PATH_OVERRIDE
        if GLOBAL_PROTOCOL_PATH_OVERRIDE is not None
        else GLOBAL_PROTOCOL_PATH
    ),
    drive_root=DRIVE_SEARCH_ROOT,
)

MASTER_MANIFEST = read_json_required(MASTER_MANIFEST_PATH, "master manifest")
TAXONOMY_MANIFEST = read_json_required(TAXONOMY_MANIFEST_PATH, "taxonomy manifest")
ROW_IDENTITY_MANIFEST = read_json_required(
    ROW_IDENTITY_MANIFEST_PATH, "row identity manifest"
)
RAW_FEATURE_POLICY = read_json_required(
    RAW_FEATURE_POLICY_PATH, "raw feature policy"
)
FINGERPRINT_AUDIT = read_json_required(
    FINGERPRINT_AUDIT_PATH, "fingerprint audit"
)
GLOBAL_PROTOCOL = read_json_required(
    GLOBAL_PROTOCOL_PATH, "global protocol"
)

print("Resolved validated CICIDS2017 Sections 1–7 root:", UPSTREAM_DIR)
print("Resolved readiness report:", PREPARTITION_READINESS_REPORT_PATH)
print("Resolved frozen master:", MASTER_DATASET_PATH)
print("Resolved global protocol:", GLOBAL_PROTOCOL_PATH)

# Master identity.
MASTER_SHA256 = sha256_file(MASTER_DATASET_PATH)
expected_master_hashes = {
    str(READINESS.get("master_dataset_sha256", "")).lower(),
    str(MASTER_MANIFEST.get("master_sha256", "")).lower(),
    str(RAW_FEATURE_POLICY.get("master_dataset_sha256", "")).lower(),
    str(TAXONOMY_MANIFEST.get("master_dataset_sha256", "")).lower(),
    str(FINGERPRINT_AUDIT.get("master_dataset_sha256", "")).lower(),
}
expected_master_hashes.discard("")
if expected_master_hashes != {MASTER_SHA256.lower()}:
    raise RuntimeError(
        "Master dataset SHA-256 mismatch across upstream artifacts. "
        f"actual={MASTER_SHA256}, expected_set={sorted(expected_master_hashes)}"
    )

# Global protocol hash.
global_hash_computed = canonical_hash_from_json_field(
    GLOBAL_PROTOCOL, "canonical_protocol_hash"
)
global_hash_stored = str(GLOBAL_PROTOCOL.get("canonical_protocol_hash", "")).lower()
global_hash_readiness = str(READINESS.get("global_protocol_hash", "")).lower()
if not (
    global_hash_computed.lower()
    == global_hash_stored
    == global_hash_readiness
):
    raise RuntimeError(
        "Global protocol hash mismatch. "
        f"computed={global_hash_computed}, stored={global_hash_stored}, "
        f"readiness={global_hash_readiness}"
    )
GLOBAL_PROTOCOL_HASH = global_hash_computed

# Taxonomy.
TAXONOMY_HASH = str(TAXONOMY_MANIFEST.get("taxonomy_hash", "")).lower()
if TAXONOMY_HASH != str(READINESS.get("taxonomy_hash", "")).lower():
    raise RuntimeError("Taxonomy hash mismatch between manifest and readiness report.")

# Row identity.
ROW_ID_SPEC_HASH = str(
    ROW_IDENTITY_MANIFEST.get("row_identity_specification_hash", "")
).lower()
if ROW_ID_SPEC_HASH != str(
    READINESS.get("row_identity_specification_hash", "")
).lower():
    raise RuntimeError(
        "Row identity specification hash mismatch between manifest and readiness."
    )

# Raw feature policy.
RAW_FEATURE_POLICY_HASH = str(
    RAW_FEATURE_POLICY.get("feature_policy_hash", "")
).lower()
if RAW_FEATURE_POLICY_HASH != str(
    READINESS.get("raw_feature_policy_hash", "")
).lower():
    raise RuntimeError(
        "Raw feature-policy hash mismatch between manifest and readiness."
    )

# Fingerprint mapping is the authoritative partitioning input.
FINGERPRINT_MANIFEST_FILE_SHA256 = sha256_file(FINGERPRINT_ROW_MAPPING_PATH)
if FINGERPRINT_MANIFEST_FILE_SHA256.lower() != str(
    READINESS.get("fingerprint_manifest_hash", "")
).lower():
    raise RuntimeError(
        "Fingerprint row-mapping SHA-256 mismatch. "
        f"actual={FINGERPRINT_MANIFEST_FILE_SHA256}, "
        f"expected={READINESS.get('fingerprint_manifest_hash')}"
    )

if str(FINGERPRINT_AUDIT.get("master_dataset_sha256", "")).lower() != MASTER_SHA256.lower():
    raise RuntimeError("Fingerprint audit is bound to a different master dataset.")

MASTER_ROW_COUNT = int(MASTER_MANIFEST.get("row_count", -1))
if MASTER_ROW_COUNT <= 0:
    raise RuntimeError("Master manifest has no valid row_count.")

if PARTITION_SEED is None or isinstance(PARTITION_SEED, bool):
    raise RuntimeError(
        "PARTITION_SEED must be a prospectively frozen explicit integer."
    )
try:
    PARTITION_SEED = int(PARTITION_SEED)
except Exception as exc:
    raise RuntimeError("PARTITION_SEED must be an integer.") from exc

if not (0.0 < MAX_MIXED_ATTACK_PREVALENCE < 0.5):
    raise RuntimeError("MAX_MIXED_ATTACK_PREVALENCE must be >0 and strictly <0.5.")

UPSTREAM_HASHES = {
    "master_dataset_sha256": MASTER_SHA256,
    "global_protocol_hash": GLOBAL_PROTOCOL_HASH,
    "taxonomy_hash": TAXONOMY_HASH,
    "row_identity_specification_hash": ROW_ID_SPEC_HASH,
    "raw_feature_policy_hash": RAW_FEATURE_POLICY_HASH,
    "fingerprint_manifest_hash": FINGERPRINT_MANIFEST_FILE_SHA256,
}

print("PASS: Sections 1–7 readiness and upstream hashes are mutually consistent.")
display(pd.DataFrame(
    [{"upstream_identity": k, "hash": v} for k, v in UPSTREAM_HASHES.items()]
))


Resolved validated CICIDS2017 Sections 1–7 root: /content/exp1v2_google_drive/MyDrive/EXPERIMENT1_V2/DATASETS/CICIDS2017/CICIDS_Audit_Governance
Resolved readiness report: /content/exp1v2_google_drive/MyDrive/EXPERIMENT1_V2/DATASETS/CICIDS2017/CICIDS_Audit_Governance/readiness/prepartition_readiness_report.json
Resolved frozen master: /content/exp1v2_google_drive/MyDrive/EXPERIMENT1_V2/DATASETS/CICIDS2017/MERGED/CICIDS2017_MachineLearningCSV_merged_raw.csv
Resolved global protocol: /content/exp1v2_google_drive/MyDrive/EXPERIMENT1_V2/global/experiment1_v2_global_protocol.json
PASS: Sections 1–7 readiness and upstream hashes are mutually consistent.


,upstream_identity,hash
0,master_dataset_sha256,b68a108f6fe44082a051e2ab81a4bd8859cb3ccd417ff6...
1,global_protocol_hash,55713fd82c01168e0a906d282644b1c8dff88632dcb825...
2,taxonomy_hash,e4290a4c91d69236e84f294fdadde7eda7f94aaa7dd6c3...
3,row_identity_specification_hash,75d07ee359615a65b65107ba4999df581e382e13395916...
4,raw_feature_policy_hash,3cae0d0bb613b1966f2542c2211afadcf4cdc2bcb6ceb9...
5,fingerprint_manifest_hash,1bc46e61bd3c74df45cd21cf280b773693a66898f3a53b...


# Load authoritative row/fingerprint metadata and validate group classes


In [ ]:
# ============================================================
# 9–10. LOAD AUTHORITATIVE ROW/FINGERPRINT METADATA AND
#       VALIDATE FINGERPRINT-GROUP CLASSES
# ============================================================
mapping_columns = [
    "row_id",
    "feature_fingerprint",
    "fingerprint_group_size",
    "fingerprint_group_class",
    "binary_label",
    "attack_family_metadata",
    "future_partition_eligibility",
]

ROW_MAP = pd.read_parquet(
    FINGERPRINT_ROW_MAPPING_PATH,
    columns=mapping_columns,
    engine="pyarrow",
)

if len(ROW_MAP) != MASTER_ROW_COUNT:
    raise RuntimeError(
        f"Fingerprint mapping rows {len(ROW_MAP):,} != master rows {MASTER_ROW_COUNT:,}."
    )
if ROW_MAP["row_id"].isna().any() or ROW_MAP["row_id"].duplicated().any():
    raise RuntimeError("row_id is null or non-unique in the fingerprint mapping.")
if ROW_MAP["feature_fingerprint"].isna().any():
    raise RuntimeError("feature_fingerprint contains null values.")
if not set(ROW_MAP["binary_label"].dropna().unique()).issubset({0, 1}):
    raise RuntimeError("binary_label contains values outside {0,1}.")

ROW_MAP["binary_label"] = ROW_MAP["binary_label"].astype("int8")
ROW_MAP["_benign_row"] = (ROW_MAP["binary_label"] == 0).astype("int32")
ROW_MAP["_attack_row"] = (ROW_MAP["binary_label"] == 1).astype("int32")

GROUPS = (
    ROW_MAP.groupby("feature_fingerprint", sort=False)
    .agg(
        total_rows=("row_id", "size"),
        benign_rows=("_benign_row", "sum"),
        attack_rows=("_attack_row", "sum"),
        upstream_group_class=("fingerprint_group_class", "first"),
        upstream_group_class_nunique=("fingerprint_group_class", "nunique"),
    )
    .reset_index()
)

if (GROUPS["upstream_group_class_nunique"] != 1).any():
    raise RuntimeError(
        "At least one feature_fingerprint has inconsistent upstream group classes."
    )

GROUPS["allocation_group_class"] = np.select(
    [
        GROUPS["attack_rows"].eq(0),
        GROUPS["benign_rows"].eq(0),
        GROUPS["benign_rows"].gt(0) & GROUPS["attack_rows"].gt(0),
    ],
    [
        "pure benign group",
        "pure attack group",
        "mixed-label/conflicting group",
    ],
    default="UNRESOLVED",
)

if GROUPS["allocation_group_class"].eq("UNRESOLVED").any():
    raise RuntimeError("At least one fingerprint group could not be classified.")

upstream_mixed = GROUPS["upstream_group_class"].eq(
    "mixed-label/conflicting group"
)
derived_mixed = GROUPS["allocation_group_class"].eq(
    "mixed-label/conflicting group"
)
if not bool((upstream_mixed == derived_mixed).all()):
    bad = GROUPS.loc[
        upstream_mixed != derived_mixed,
        [
            "feature_fingerprint",
            "upstream_group_class",
            "allocation_group_class",
            "benign_rows",
            "attack_rows",
        ],
    ].head(20)
    raise RuntimeError(
        "Derived mixed-label classification materially disagrees with Sections 1–7.\n"
        + bad.to_string(index=False)
    )

# Validate upstream group_size field.
observed_group_size = ROW_MAP.groupby(
    "feature_fingerprint", sort=False
)["row_id"].transform("size")
if not bool(
    (observed_group_size.to_numpy()
     == ROW_MAP["fingerprint_group_size"].to_numpy()).all()
):
    raise RuntimeError(
        "Upstream fingerprint_group_size is inconsistent with the row mapping."
    )
del observed_group_size

# Attack-family metadata must exist for every attack row.
attack_family = ROW_MAP.loc[
    ROW_MAP["binary_label"].eq(1), "attack_family_metadata"
].astype("string")
invalid_family = attack_family.isna() | attack_family.str.strip().eq("")
if invalid_family.any():
    examples = ROW_MAP.loc[
        ROW_MAP["binary_label"].eq(1) & invalid_family.reindex(ROW_MAP.index, fill_value=False),
        ["row_id", "feature_fingerprint", "attack_family_metadata"],
    ].head(20)
    raise RuntimeError(
        "Attack rows with missing/blank authoritative family metadata were found.\n"
        + examples.to_string(index=False)
    )

B_TOTAL = int(ROW_MAP["_benign_row"].sum())
A_TOTAL = int(ROW_MAP["_attack_row"].sum())
if B_TOTAL + A_TOTAL != MASTER_ROW_COUNT:
    raise RuntimeError("Binary class accounting does not sum to master row count.")

# Attack family × fingerprint support table.
ATTACK_ROWS = ROW_MAP.loc[
    ROW_MAP["binary_label"].eq(1),
    ["feature_fingerprint", "attack_family_metadata"],
].copy()
ATTACK_ROWS["attack_family_metadata"] = (
    ATTACK_ROWS["attack_family_metadata"].astype("string")
)

FAMILY_GROUP_COUNTS = (
    ATTACK_ROWS.groupby(
        ["feature_fingerprint", "attack_family_metadata"],
        sort=False,
        dropna=False,
    )
    .size()
    .rename("family_rows_in_group")
    .reset_index()
)

FAMILY_TOTALS = (
    ATTACK_ROWS.groupby("attack_family_metadata", sort=False)
    .size()
    .rename("total_row_support")
    .reset_index()
)

FAMILY_GROUP_SUPPORT = (
    ATTACK_ROWS.groupby("attack_family_metadata", sort=False)[
        "feature_fingerprint"
    ]
    .nunique()
    .rename("independent_fingerprint_group_support")
    .reset_index()
)

FAMILY_BASE = FAMILY_TOTALS.merge(
    FAMILY_GROUP_SUPPORT,
    on="attack_family_metadata",
    how="left",
    validate="one_to_one",
)

FAMILY_GROUP_SUPPORT_MAP = dict(zip(
    FAMILY_BASE["attack_family_metadata"].astype(str),
    FAMILY_BASE["independent_fingerprint_group_support"].astype(int),
))

print("Frozen row/fingerprint metadata loaded successfully.")
display(pd.DataFrame([{
    "rows": MASTER_ROW_COUNT,
    "benign_rows": B_TOTAL,
    "attack_rows": A_TOTAL,
    "fingerprint_groups": len(GROUPS),
    "pure_benign_groups": int(
        GROUPS["allocation_group_class"].eq("pure benign group").sum()
    ),
    "pure_attack_groups": int(
        GROUPS["allocation_group_class"].eq("pure attack group").sum()
    ),
    "mixed_label_groups": int(
        GROUPS["allocation_group_class"].eq(
            "mixed-label/conflicting group"
        ).sum()
    ),
    "attack_families": int(len(FAMILY_BASE)),
}]))


Frozen row/fingerprint metadata loaded successfully.


,rows,benign_rows,attack_rows,fingerprint_groups,pure_benign_groups,pure_attack_groups,mixed_label_groups,attack_families
0,2830743,2273097,557646,2521664,2095786,425180,698,14


# Section 8 — Partition Design Freeze


In [ ]:
# ============================================================
# 11–13. SECTION 8 — PARTITION DESIGN FREEZE
# ============================================================
BENIGN_TARGET_FRACTIONS = {
    "benign_weight_train": 0.50,
    "benign_early_stop": 0.10,
    "benign_threshold_calibration": 0.15,
    "development": 0.10,
    "final_test": 0.15,
}

BENIGN_TARGETS = benign_target_table(B_TOTAL)

ALLOCATION_ALGORITHM_VERSION = "Experiment1_v2_group_partition_v1"
TIE_BREAKING_RULE = (
    "first 64 bits of SHA256(str(PARTITION_SEED) + '|' + feature_fingerprint); "
    "full SHA-256 specification is deterministic and Python hash() is never used"
)

LEXICOGRAPHIC_PRIORITY_ORDER = [
    "1. no row/fingerprint leakage",
    "2. no attacks in benign-only roles",
    "3. strict benign majority in development and final test",
    "4. all benign observations used exactly once",
    "5. maximize documented attacks used in development + final test",
    "6. preserve rare-family evaluability",
    "7. minimize development/final-test prevalence difference",
    "8. preserve attack-family proportions",
    "9. minimize deviations from nominal benign percentages",
    "10. deterministic tie-breaking",
]

PARTITION_POLICY_SUBSTANTIVE = {
    "dataset": DATASET_NAME,
    "protocol_version": PROTOCOL_VERSION,
    "PARTITION_SEED": int(PARTITION_SEED),
    "allocation_algorithm_version": ALLOCATION_ALGORITHM_VERSION,
    "allocation_method": (
        "deterministic fingerprint-group heuristic with hard post-allocation "
        "integrity validation; no claim of global combinatorial optimality"
    ),
    "tie_breaking_rule": TIE_BREAKING_RULE,
    "allocation_unit": "feature_fingerprint group",
    "benign_target_fractions": BENIGN_TARGET_FRACTIONS,
    "MAX_MIXED_ATTACK_PREVALENCE": float(MAX_MIXED_ATTACK_PREVALENCE),
    "lexicographic_priority_order": LEXICOGRAPHIC_PRIORITY_ORDER,
    "family_allocation_rule": (
        "deterministic proportional largest-remainder row targets; actual allocation "
        "uses indivisible fingerprint groups and records deviations"
    ),
    "rare_family_rule": (
        "judge independence by fingerprint-group support; one-group families prefer "
        "final_test when higher-priority constraints permit; never duplicate/split"
    ),
    "mixed_label_group_rule": (
        "mixed-label groups remain intact and may enter development/final_test only"
    ),
    "attack_reserve_rule": (
        "optional governance state containing only unavoidable excess pure-attack "
        "groups; no benign reserve"
    ),
    "final_test_sealing_rule": (
        "membership/labels/family metadata frozen after integrity gate; final-test "
        "feature transformation later is permitted with a benign_weight_train-fitted "
        "frozen preprocessor, but final-test influence is forbidden before Notebook 07"
    ),
    "preprocessing_handoff_rule": {
        "fit": ["benign_weight_train"],
        "transform": EXPERIMENTAL_PARTITIONS,
        "final_test_rule": "transformation permitted; influence forbidden",
    },
    "upstream_hashes": UPSTREAM_HASHES,
}

PARTITION_POLICY_HASH = sha256_bytes(
    canonical_json_bytes(PARTITION_POLICY_SUBSTANTIVE)
)

PARTITION_POLICY_PATH = OUTPUT_DIR / "partition_policy_manifest.json"

# Reuse an already-frozen creation timestamp only when the policy hash matches.
if PARTITION_POLICY_PATH.exists():
    existing_policy = read_json_required(
        PARTITION_POLICY_PATH, "existing partition policy"
    )
    if existing_policy.get("partition_policy_hash") != PARTITION_POLICY_HASH:
        raise RuntimeError(
            "A conflicting partition policy already exists in OUTPUT_DIR. "
            "Use a new versioned output directory rather than overwriting it."
        )
    PARTITION_CREATED_UTC = existing_policy.get("created_at_utc")
else:
    PARTITION_CREATED_UTC = EXECUTION_UTC

PARTITION_POLICY = {
    **PARTITION_POLICY_SUBSTANTIVE,
    "created_at_utc": PARTITION_CREATED_UTC,
    "partition_policy_hash": PARTITION_POLICY_HASH,
}

write_json_frozen(PARTITION_POLICY_PATH, PARTITION_POLICY)

display(pd.DataFrame([
    {
        "role": p,
        "nominal_fraction_of_all_benign": BENIGN_TARGET_FRACTIONS[p],
        "nominal_benign_rows": BENIGN_TARGETS[p],
    }
    for p in EXPERIMENTAL_PARTITIONS
]))

print("PARTITION_SEED:", PARTITION_SEED)
print("Partition policy hash:", PARTITION_POLICY_HASH)


,role,nominal_fraction_of_all_benign,nominal_benign_rows
0,benign_weight_train,0.50,1136548
1,benign_early_stop,0.10,227309
2,benign_threshold_calibration,0.15,340964
3,development,0.10,227309
4,final_test,0.15,340967


PARTITION_SEED: 20260914
Partition policy hash: be62dd88cedd77903c19f64c3bab1d5fac3508820d67906f5f17381eddcde66a


# Assign mandatory mixed-label/conflicting groups


In [ ]:
# ============================================================
# 14. ASSIGN MIXED-LABEL GROUPS SUBJECT TO HARD ROLE CONSTRAINTS
# ============================================================
GROUP_ASSIGNMENT = {}
RESERVE_REASON = {}
MIXED_ASSIGNMENT_ROWS = []

MIXED_GROUPS = GROUPS.loc[
    GROUPS["allocation_group_class"].eq("mixed-label/conflicting group"),
    [
        "feature_fingerprint",
        "total_rows",
        "benign_rows",
        "attack_rows",
    ],
].copy()

# Families carried by each mixed group.
MIXED_FAMILY_MAP = {}
if len(MIXED_GROUPS):
    mixed_fp_set = set(MIXED_GROUPS["feature_fingerprint"])
    tmp = FAMILY_GROUP_COUNTS[
        FAMILY_GROUP_COUNTS["feature_fingerprint"].isin(mixed_fp_set)
    ]
    for fp, sub in tmp.groupby("feature_fingerprint", sort=False):
        MIXED_FAMILY_MAP[fp] = {
            str(r.attack_family_metadata): int(r.family_rows_in_group)
            for r in sub.itertuples(index=False)
        }
    del tmp

MIXED_GROUPS["contains_singleton_family"] = MIXED_GROUPS[
    "feature_fingerprint"
].map(
    lambda fp: any(
        FAMILY_GROUP_SUPPORT_MAP.get(fam, 0) == 1
        for fam in MIXED_FAMILY_MAP.get(fp, {})
    )
)
MIXED_GROUPS["_stable_key"] = MIXED_GROUPS["feature_fingerprint"].map(
    lambda fp: stable_key64(PARTITION_SEED, fp)
)

# Rare singleton-family evaluability is prioritized within the mixed-group step.
MIXED_GROUPS = MIXED_GROUPS.sort_values(
    ["contains_singleton_family", "total_rows", "_stable_key"],
    ascending=[False, False, True],
    kind="mergesort",
)

mixed_counts = {
    "development": {"benign": 0, "attack": 0},
    "final_test": {"benign": 0, "attack": 0},
}

nominal_mixed_caps = {
    p: attack_capacity_from_benign(
        BENIGN_TARGETS[p], MAX_MIXED_ATTACK_PREVALENCE
    )
    for p in MIXED_EVALUATION_PARTITIONS
}

for row in MIXED_GROUPS.itertuples(index=False):
    fp = row.feature_fingerprint
    gB = int(row.benign_rows)
    gA = int(row.attack_rows)

    candidates = []
    for partition in MIXED_EVALUATION_PARTITIONS:
        other = (
            "final_test" if partition == "development" else "development"
        )

        new_B_this = mixed_counts[partition]["benign"] + gB
        new_A_this = mixed_counts[partition]["attack"] + gA

        # Soft score only. Final hard prevalence is validated after pure-benign fill.
        benign_target_cost = (
            abs(new_B_this - BENIGN_TARGETS[partition])
            / max(1, BENIGN_TARGETS[partition])
            + abs(
                mixed_counts[other]["benign"] - BENIGN_TARGETS[other]
            )
            / max(1, BENIGN_TARGETS[other])
        )

        nominal_cap_violation = max(
            0, new_A_this - nominal_mixed_caps[partition]
        )

        singleton_penalty = (
            0
            if (not row.contains_singleton_family or partition == "final_test")
            else 1
        )

        candidates.append((
            singleton_penalty,
            nominal_cap_violation,
            benign_target_cost,
            0 if partition == "final_test" else 1,
            partition,
        ))

    chosen = min(candidates)[-1]
    GROUP_ASSIGNMENT[fp] = chosen
    mixed_counts[chosen]["benign"] += gB
    mixed_counts[chosen]["attack"] += gA

    MIXED_ASSIGNMENT_ROWS.append({
        "feature_fingerprint": fp,
        "partition": chosen,
        "benign_rows": gB,
        "attack_rows": gA,
        "contains_singleton_family": bool(row.contains_singleton_family),
        "family_vector_json": json.dumps(
            MIXED_FAMILY_MAP.get(fp, {}),
            sort_keys=True,
            ensure_ascii=False,
        ),
        "stable_tie_key": stable_hex(PARTITION_SEED, fp),
    })

MIXED_ASSIGNMENT_DF = pd.DataFrame(MIXED_ASSIGNMENT_ROWS)

print("Mixed-label mandatory assignments:")
display(pd.DataFrame([
    {
        "partition": p,
        "mixed_group_benign_rows": mixed_counts[p]["benign"],
        "mixed_group_attack_rows": mixed_counts[p]["attack"],
        "nominal_attack_cap_at_nominal_benign_target": nominal_mixed_caps[p],
    }
    for p in MIXED_EVALUATION_PARTITIONS
]))


Mixed-label mandatory assignments:


,partition,mixed_group_benign_rows,mixed_group_attack_rows,nominal_attack_cap_at_nominal_benign_target
0,development,736,6284,218394
1,final_test,0,0,327595


# Allocate pure-benign fingerprint groups


In [ ]:
# ============================================================
# 15–16. ALLOCATE PURE-BENIGN GROUPS AND DERIVE ACHIEVED BENIGN COUNTS
# ============================================================
current_benign = {
    "benign_weight_train": 0,
    "benign_early_stop": 0,
    "benign_threshold_calibration": 0,
    "development": int(mixed_counts["development"]["benign"]),
    "final_test": int(mixed_counts["final_test"]["benign"]),
}

PURE_BENIGN = GROUPS.loc[
    GROUPS["allocation_group_class"].eq("pure benign group"),
    ["feature_fingerprint", "benign_rows"],
].copy()

PURE_BENIGN["_stable_key"] = PURE_BENIGN["feature_fingerprint"].map(
    lambda fp: stable_key64(PARTITION_SEED, fp)
)

# Multi-row benign groups are the source of atomic row-count deviations.
BENIGN_MULTI = PURE_BENIGN.loc[PURE_BENIGN["benign_rows"].gt(1)].copy()
BENIGN_UNIT = PURE_BENIGN.loc[PURE_BENIGN["benign_rows"].eq(1)].copy()

BENIGN_MULTI = BENIGN_MULTI.sort_values(
    ["benign_rows", "_stable_key"],
    ascending=[False, True],
    kind="mergesort",
)

for row in BENIGN_MULTI.itertuples(index=False):
    size = int(row.benign_rows)

    scored = []
    for partition in EXPERIMENTAL_PARTITIONS:
        cost = normalized_incremental_cost(
            current_benign[partition],
            BENIGN_TARGETS[partition],
            size,
        )
        scored.append((
            cost,
            abs(
                (current_benign[partition] + size)
                - BENIGN_TARGETS[partition]
            ),
            EXPERIMENTAL_PARTITIONS.index(partition),
            partition,
        ))

    chosen = min(scored)[-1]
    GROUP_ASSIGNMENT[row.feature_fingerprint] = chosen
    current_benign[chosen] += size

# Unit groups can fill remaining positive row deficits exactly.
BENIGN_UNIT = BENIGN_UNIT.sort_values(
    "_stable_key", ascending=True, kind="mergesort"
).reset_index(drop=True)

unit_cursor = 0
for partition in EXPERIMENTAL_PARTITIONS:
    deficit = max(
        0, int(BENIGN_TARGETS[partition] - current_benign[partition])
    )
    available = len(BENIGN_UNIT) - unit_cursor
    take = min(deficit, available)

    if take > 0:
        fps = BENIGN_UNIT.iloc[
            unit_cursor:unit_cursor + take
        ]["feature_fingerprint"].tolist()
        for fp in fps:
            GROUP_ASSIGNMENT[fp] = partition
        current_benign[partition] += take
        unit_cursor += take

# If mixed/multi-row atomic groups already overfilled some targets, there may be
# unit groups left after all positive deficits are closed. Place them one by one
# in the role with the smallest normalized incremental deviation.
if unit_cursor < len(BENIGN_UNIT):
    leftovers = BENIGN_UNIT.iloc[unit_cursor:]
    for row in leftovers.itertuples(index=False):
        scored = []
        for partition in EXPERIMENTAL_PARTITIONS:
            cost = normalized_incremental_cost(
                current_benign[partition],
                BENIGN_TARGETS[partition],
                1,
            )
            scored.append((
                cost,
                EXPERIMENTAL_PARTITIONS.index(partition),
                partition,
            ))
        chosen = min(scored)[-1]
        GROUP_ASSIGNMENT[row.feature_fingerprint] = chosen
        current_benign[chosen] += 1

BENIGN_ACHIEVED = {k: int(v) for k, v in current_benign.items()}

if sum(BENIGN_ACHIEVED.values()) != B_TOTAL:
    raise RuntimeError(
        "Benign group allocation does not account for every benign observation."
    )

# No pure-benign fingerprint may remain unassigned.
missing_benign_groups = [
    fp for fp in PURE_BENIGN["feature_fingerprint"]
    if fp not in GROUP_ASSIGNMENT
]
if missing_benign_groups:
    raise RuntimeError(
        f"{len(missing_benign_groups)} pure-benign groups remain unassigned."
    )

BENIGN_ALLOCATION_ROWS = []
for partition in EXPERIMENTAL_PARTITIONS:
    target = int(BENIGN_TARGETS[partition])
    actual = int(BENIGN_ACHIEVED[partition])
    nominal_pct = 100.0 * target / B_TOTAL if B_TOTAL else np.nan
    actual_pct = 100.0 * actual / B_TOTAL if B_TOTAL else np.nan
    deviation = actual - target

    if deviation == 0:
        reason = "EXACT_TARGET"
    else:
        reason = (
            "ATOMIC_FINGERPRINT_GROUP_INDIVISIBILITY_AND_OR_"
            "MANDATORY_MIXED_LABEL_GROUP_PLACEMENT"
        )
        record_warning(
            f"{partition} benign allocation deviates from nominal by "
            f"{deviation:+,} rows due to atomic group constraints."
        )

    BENIGN_ALLOCATION_ROWS.append({
        "partition": partition,
        "nominal_benign_target": target,
        "actual_benign_count": actual,
        "absolute_row_deviation": int(abs(deviation)),
        "signed_row_deviation": int(deviation),
        "nominal_percentage_of_B": nominal_pct,
        "actual_percentage_of_B": actual_pct,
        "percentage_point_deviation": actual_pct - nominal_pct,
        "reason": reason,
    })

BENIGN_ALLOCATION_DF = pd.DataFrame(BENIGN_ALLOCATION_ROWS)

print("Achieved benign allocation:")
display(BENIGN_ALLOCATION_DF)


Achieved benign allocation:


,partition,nominal_benign_target,actual_benign_count,absolute_row_deviation,signed_row_deviation,nominal_percentage_of_B,actual_percentage_of_B,percentage_point_deviation,reason
0,benign_weight_train,1136548,1136548,0,0,49.999978,49.999978,0.0,EXACT_TARGET
1,benign_early_stop,227309,227309,0,0,9.999969,9.999969,0.0,EXACT_TARGET
2,benign_threshold_calibration,340964,340964,0,0,14.999976,14.999976,0.0,EXACT_TARGET
3,development,227309,227309,0,0,9.999969,9.999969,0.0,EXACT_TARGET
4,final_test,340967,340967,0,0,15.000108,15.000108,0.0,EXACT_TARGET


# Section 9 — Attack capacity, maximum use, and family targets


In [ ]:
# ============================================================
# 17–19. SECTION 9 — HARD ATTACK CAPACITIES, MAXIMUM ATTACK USE,
#        AND FAMILY TARGETS
# ============================================================
B_DEV = int(BENIGN_ACHIEVED["development"])
B_TEST = int(BENIGN_ACHIEVED["final_test"])

if B_DEV <= 0 or B_TEST <= 0:
    raise RuntimeError(
        "Development/final-test benign counts must both be positive."
    )

A_DEV_CAP = attack_capacity_from_benign(
    B_DEV, MAX_MIXED_ATTACK_PREVALENCE
)
A_TEST_CAP = attack_capacity_from_benign(
    B_TEST, MAX_MIXED_ATTACK_PREVALENCE
)

MIXED_A_DEV = int(mixed_counts["development"]["attack"])
MIXED_A_TEST = int(mixed_counts["final_test"]["attack"])

if MIXED_A_DEV > A_DEV_CAP or MIXED_A_TEST > A_TEST_CAP:
    raise RuntimeError(
        "Mandatory mixed-label groups alone violate a hard benign-majority attack "
        "capacity after achieved benign allocation. No valid partition exists under "
        "the frozen policy; do not move mixed groups to reserve."
    )

A_CAPACITY_TOTAL = int(A_DEV_CAP + A_TEST_CAP)
A_USE_TARGET = int(min(A_TOTAL, A_CAPACITY_TOTAL))

q_dev = B_DEV / (B_DEV + B_TEST)
q_test = 1.0 - q_dev

raw_dev_target = int(math.floor(q_dev * A_USE_TARGET))
A_DEV_TARGET, A_TEST_TARGET = adjust_attack_targets(
    A_USE_TARGET,
    raw_dev_target,
    A_DEV_CAP,
    A_TEST_CAP,
    MIXED_A_DEV,
    MIXED_A_TEST,
)

# Family row targets are guidance only; atomic fingerprint groups outrank them.
family_weights = {
    str(r.attack_family_metadata): int(r.total_row_support)
    for r in FAMILY_BASE.itertuples(index=False)
}
family_names = sorted(family_weights)

dev_weights = {
    fam: (
        family_weights[fam]
        if FAMILY_GROUP_SUPPORT_MAP.get(fam, 0) >= 2
        else 0
    )
    for fam in family_names
}
if sum(dev_weights.values()) <= 0:
    dev_weights = dict(family_weights)

FAMILY_DEV_TARGET = largest_remainder(
    A_DEV_TARGET,
    dev_weights,
    tie_order=family_names,
)
FAMILY_TEST_TARGET = largest_remainder(
    A_TEST_TARGET,
    family_weights,
    tie_order=family_names,
)

FAMILY_TARGET_ROWS = pd.DataFrame([
    {
        "family": fam,
        "total_row_support": family_weights[fam],
        "independent_fingerprint_group_support": int(
            FAMILY_GROUP_SUPPORT_MAP.get(fam, 0)
        ),
        "target_development_rows": int(FAMILY_DEV_TARGET[fam]),
        "target_final_test_rows": int(FAMILY_TEST_TARGET[fam]),
        "target_selected_rows": int(
            FAMILY_DEV_TARGET[fam] + FAMILY_TEST_TARGET[fam]
        ),
    }
    for fam in family_names
])

print("Attack capacity/target summary:")
display(pd.DataFrame([{
    "B_dev": B_DEV,
    "B_test": B_TEST,
    "A_dev_cap": A_DEV_CAP,
    "A_test_cap": A_TEST_CAP,
    "A_capacity_total": A_CAPACITY_TOTAL,
    "A_total": A_TOTAL,
    "A_use_target": A_USE_TARGET,
    "q_dev": q_dev,
    "q_test": q_test,
    "A_dev_target": A_DEV_TARGET,
    "A_test_target": A_TEST_TARGET,
    "mandatory_mixed_A_dev": MIXED_A_DEV,
    "mandatory_mixed_A_test": MIXED_A_TEST,
}]))
display(FAMILY_TARGET_ROWS)


Attack capacity/target summary:


,B_dev,B_test,A_dev_cap,A_test_cap,A_capacity_total,A_total,A_use_target,q_dev,q_test,A_dev_target,A_test_target,mandatory_mixed_A_dev,mandatory_mixed_A_test
0,227309,340967,218394,327595,545989,557646,545989,0.399998,0.600002,218394,327595,6284,0


,family,total_row_support,independent_fingerprint_group_support,target_development_rows,target_final_test_rows,target_selected_rows
0,Bot,1966,1953,770,1155,1925
1,DDoS,128027,128016,50140,75211,125351
2,DoS GoldenEye,10293,10286,4031,6047,10078
3,DoS Hulk,231073,172849,90496,135746,226242
4,DoS Slowhttptest,5499,5228,2154,3231,5385
5,DoS slowloris,5796,5385,2270,3405,5675
6,FTP-Patator,7938,5933,3109,4663,7772
7,Heartbleed,11,11,4,7,11
8,Infiltration,36,36,14,21,35
9,PortScan,158930,90819,62243,93365,155608


# Allocate pure-attack groups and construct optional reserve


In [ ]:
# ============================================================
# 20–21. ALLOCATE PURE-ATTACK FINGERPRINT GROUPS AND BUILD RESERVE
# ============================================================
PURE_ATTACK = GROUPS.loc[
    GROUPS["allocation_group_class"].eq("pure attack group"),
    ["feature_fingerprint", "attack_rows"],
].copy()

# Family vector for each pure-attack group.
PURE_ATTACK_FP_SET = set(PURE_ATTACK["feature_fingerprint"])
pure_family_counts = FAMILY_GROUP_COUNTS[
    FAMILY_GROUP_COUNTS["feature_fingerprint"].isin(PURE_ATTACK_FP_SET)
].copy()

family_nunique_by_fp = (
    pure_family_counts.groupby("feature_fingerprint", sort=False)[
        "attack_family_metadata"
    ]
    .nunique()
    .rename("family_nunique")
)

single_family_by_fp = (
    pure_family_counts.sort_values(
        ["feature_fingerprint", "attack_family_metadata"],
        kind="mergesort",
    )
    .groupby("feature_fingerprint", sort=False)[
        "attack_family_metadata"
    ]
    .first()
)

PURE_ATTACK = PURE_ATTACK.merge(
    family_nunique_by_fp,
    left_on="feature_fingerprint",
    right_index=True,
    how="left",
    validate="one_to_one",
)
PURE_ATTACK = PURE_ATTACK.merge(
    single_family_by_fp.rename("first_family"),
    left_on="feature_fingerprint",
    right_index=True,
    how="left",
    validate="one_to_one",
)

PURE_ATTACK["family_nunique"] = PURE_ATTACK[
    "family_nunique"
].fillna(0).astype("int32")

# Multi-family vectors are retained only for groups that need them.
MULTI_FAMILY_VECTOR = {}
multi_fp = set(
    PURE_ATTACK.loc[
        PURE_ATTACK["family_nunique"].gt(1),
        "feature_fingerprint",
    ].tolist()
)
if multi_fp:
    for fp, sub in pure_family_counts[
        pure_family_counts["feature_fingerprint"].isin(multi_fp)
    ].groupby("feature_fingerprint", sort=False):
        MULTI_FAMILY_VECTOR[fp] = {
            str(r.attack_family_metadata): int(r.family_rows_in_group)
            for r in sub.itertuples(index=False)
        }

def family_vector_for_pure_group(fp, size, first_family, family_nunique):
    if int(family_nunique) <= 1:
        return {str(first_family): int(size)}
    return dict(MULTI_FAMILY_VECTOR[fp])

PURE_ATTACK["contains_singleton_family"] = PURE_ATTACK.apply(
    lambda r: any(
        FAMILY_GROUP_SUPPORT_MAP.get(fam, 0) == 1
        for fam in family_vector_for_pure_group(
            r["feature_fingerprint"],
            r["attack_rows"],
            r["first_family"],
            r["family_nunique"],
        )
    ),
    axis=1,
)
PURE_ATTACK["_stable_key"] = PURE_ATTACK["feature_fingerprint"].map(
    lambda fp: stable_key64(PARTITION_SEED, fp)
)

attack_current = {
    "development": MIXED_A_DEV,
    "final_test": MIXED_A_TEST,
}

family_current = {
    "development": Counter(),
    "final_test": Counter(),
}
for fp, partition in [
    (r["feature_fingerprint"], r["partition"])
    for r in MIXED_ASSIGNMENT_ROWS
]:
    for fam, n in MIXED_FAMILY_MAP.get(fp, {}).items():
        family_current[partition][fam] += int(n)

# Priority groups: any multi-row group, multi-family group, or group carrying a
# one-independent-group family. Unit single-family groups are handled efficiently later.
PRIORITY_ATTACK = PURE_ATTACK.loc[
    PURE_ATTACK["attack_rows"].gt(1)
    | PURE_ATTACK["family_nunique"].gt(1)
    | PURE_ATTACK["contains_singleton_family"]
].copy()

UNIT_ATTACK = PURE_ATTACK.loc[
    PURE_ATTACK["attack_rows"].eq(1)
    & PURE_ATTACK["family_nunique"].eq(1)
    & ~PURE_ATTACK["contains_singleton_family"]
].copy()

PRIORITY_ATTACK = PRIORITY_ATTACK.sort_values(
    ["contains_singleton_family", "attack_rows", "_stable_key"],
    ascending=[False, False, True],
    kind="mergesort",
)

def placement_score(partition, size, fam_vector, rare_prefer_final):
    other = "final_test" if partition == "development" else "development"
    cap = A_DEV_CAP if partition == "development" else A_TEST_CAP

    if attack_current[partition] + size > cap:
        return None

    # Priority 6: singleton rare-family evaluability prefers final_test.
    rare_penalty = int(bool(rare_prefer_final and partition != "final_test"))

    # Priority 7: match attack totals implied by achieved benign counts.
    target_this = (
        A_DEV_TARGET if partition == "development" else A_TEST_TARGET
    )
    target_other = (
        A_TEST_TARGET if partition == "development" else A_DEV_TARGET
    )
    target_deviation = (
        abs((attack_current[partition] + size) - target_this)
        + abs(attack_current[other] - target_other)
    )

    new_dev = (
        attack_current["development"] + (size if partition == "development" else 0)
    )
    new_test = (
        attack_current["final_test"] + (size if partition == "final_test" else 0)
    )
    prevalence_gap = abs(
        new_dev / (new_dev + B_DEV)
        - new_test / (new_test + B_TEST)
    )

    # Priority 8: family target deviation.
    fam_cost = 0
    target_map = (
        FAMILY_DEV_TARGET if partition == "development" else FAMILY_TEST_TARGET
    )
    for fam, n in fam_vector.items():
        before = abs(family_current[partition][fam] - target_map.get(fam, 0))
        after = abs(
            family_current[partition][fam] + int(n) - target_map.get(fam, 0)
        )
        fam_cost += after - before

    return (
        rare_penalty,
        target_deviation,
        prevalence_gap,
        fam_cost,
        0 if partition == "final_test" else 1,
    )

for row in PRIORITY_ATTACK.itertuples(index=False):
    fp = row.feature_fingerprint
    size = int(row.attack_rows)
    fam_vector = family_vector_for_pure_group(
        fp, size, row.first_family, row.family_nunique
    )

    scored = []
    for partition in MIXED_EVALUATION_PARTITIONS:
        score = placement_score(
            partition,
            size,
            fam_vector,
            bool(row.contains_singleton_family),
        )
        if score is not None:
            scored.append((*score, partition))

    if scored:
        chosen = min(scored)[-1]
        GROUP_ASSIGNMENT[fp] = chosen
        attack_current[chosen] += size
        for fam, n in fam_vector.items():
            family_current[chosen][fam] += int(n)
    else:
        GROUP_ASSIGNMENT[fp] = "attack_reserve_unused"

        rem_dev = A_DEV_CAP - attack_current["development"]
        rem_test = A_TEST_CAP - attack_current["final_test"]

        if bool(row.contains_singleton_family):
            reason = "RARE_FAMILY_CAPACITY_CONFLICT"
        elif size > max(rem_dev, rem_test) and size <= rem_dev + rem_test:
            reason = "ATOMIC_GROUP_CAPACITY"
        else:
            reason = "BENIGN_MAJORITY_CAP"

        RESERVE_REASON[fp] = reason

# Efficient family-aware allocation of remaining unit groups.
UNIT_ATTACK = UNIT_ATTACK.sort_values(
    ["first_family", "_stable_key"],
    ascending=[True, True],
    kind="mergesort",
)

unit_leftover_indices = []

for fam, sub in UNIT_ATTACK.groupby("first_family", sort=True):
    fam = str(fam)
    indices = sub.index.to_numpy()
    cursor = 0

    for partition in ["development", "final_test"]:
        hard_cap = A_DEV_CAP if partition == "development" else A_TEST_CAP
        family_target = (
            FAMILY_DEV_TARGET.get(fam, 0)
            if partition == "development"
            else FAMILY_TEST_TARGET.get(fam, 0)
        )

        family_deficit = max(
            0, int(family_target - family_current[partition][fam])
        )
        hard_remaining = max(
            0, int(hard_cap - attack_current[partition])
        )
        take = min(
            family_deficit,
            hard_remaining,
            len(indices) - cursor,
        )

        if take > 0:
            chosen_idx = indices[cursor:cursor + take]
            fps = UNIT_ATTACK.loc[
                chosen_idx, "feature_fingerprint"
            ].tolist()
            for fp in fps:
                GROUP_ASSIGNMENT[fp] = partition
            attack_current[partition] += take
            family_current[partition][fam] += take
            cursor += take

    if cursor < len(indices):
        unit_leftover_indices.extend(indices[cursor:].tolist())

# Fill remaining hard capacity while minimizing the dev/test target/prevalence gap.
if unit_leftover_indices:
    leftovers = UNIT_ATTACK.loc[unit_leftover_indices].sort_values(
        "_stable_key", ascending=True, kind="mergesort"
    )

    for row in leftovers.itertuples(index=False):
        fp = row.feature_fingerprint
        fam = str(row.first_family)
        fam_vector = {fam: 1}

        scored = []
        for partition in MIXED_EVALUATION_PARTITIONS:
            score = placement_score(
                partition,
                1,
                fam_vector,
                False,
            )
            if score is not None:
                scored.append((*score, partition))

        if scored:
            chosen = min(scored)[-1]
            GROUP_ASSIGNMENT[fp] = chosen
            attack_current[chosen] += 1
            family_current[chosen][fam] += 1
        else:
            GROUP_ASSIGNMENT[fp] = "attack_reserve_unused"
            RESERVE_REASON[fp] = "BENIGN_MAJORITY_CAP"

# All pure-attack groups must now be accounted for.
missing_attack_groups = [
    fp for fp in PURE_ATTACK["feature_fingerprint"]
    if fp not in GROUP_ASSIGNMENT
]
if missing_attack_groups:
    raise RuntimeError(
        f"{len(missing_attack_groups)} pure-attack groups remain unassigned."
    )

A_DEV_ACHIEVED = int(attack_current["development"])
A_TEST_ACHIEVED = int(attack_current["final_test"])

# A reserved group must not fit by itself into either remaining hard-capacity bin.
reserved_fp = [
    fp for fp, p in GROUP_ASSIGNMENT.items()
    if p == "attack_reserve_unused"
]
if reserved_fp:
    pure_size_map = dict(zip(
        PURE_ATTACK["feature_fingerprint"],
        PURE_ATTACK["attack_rows"].astype(int),
    ))
    rem_dev = A_DEV_CAP - A_DEV_ACHIEVED
    rem_test = A_TEST_CAP - A_TEST_ACHIEVED

    insertable = [
        fp for fp in reserved_fp
        if pure_size_map.get(fp, 10**18) <= max(rem_dev, rem_test)
    ]
    if insertable:
        raise RuntimeError(
            "Attack-reserve allocation is not even single-group insertion maximal; "
            f"{len(insertable)} reserved groups still fit a remaining hard-capacity bin."
        )

print("Achieved attack allocation:")
display(pd.DataFrame([{
    "development_attack_target": A_DEV_TARGET,
    "development_attack_achieved": A_DEV_ACHIEVED,
    "development_attack_cap": A_DEV_CAP,
    "final_test_attack_target": A_TEST_TARGET,
    "final_test_attack_achieved": A_TEST_ACHIEVED,
    "final_test_attack_cap": A_TEST_CAP,
    "attacks_used": A_DEV_ACHIEVED + A_TEST_ACHIEVED,
    "attacks_reserved": A_TOTAL - A_DEV_ACHIEVED - A_TEST_ACHIEVED,
}]))


Achieved attack allocation:


,development_attack_target,development_attack_achieved,development_attack_cap,final_test_attack_target,final_test_attack_achieved,final_test_attack_cap,attacks_used,attacks_reserved
0,218394,218394,218394,327595,327595,327595,545989,11657


# Section 10 — Build authoritative row-level membership


In [ ]:
# ============================================================
# 22. CREATE AUTHORITATIVE ROW-LEVEL CANDIDATE MEMBERSHIP
# ============================================================
GROUP_ASSIGNMENT_SERIES = pd.Series(
    GROUP_ASSIGNMENT,
    name="partition",
    dtype="string",
)

ROW_MAP["partition"] = ROW_MAP["feature_fingerprint"].map(
    GROUP_ASSIGNMENT_SERIES
)

if ROW_MAP["partition"].isna().any():
    missing = ROW_MAP.loc[
        ROW_MAP["partition"].isna(),
        ["row_id", "feature_fingerprint"],
    ].head(20)
    raise RuntimeError(
        "Some observations have no partition membership.\n"
        + missing.to_string(index=False)
    )

if not set(ROW_MAP["partition"].unique()).issubset(
    set(ALL_MEMBERSHIP_STATES)
):
    raise RuntimeError("Unexpected membership state was generated.")

source_file, source_row = parse_row_id_fields(
    ROW_MAP["row_id"], DATASET_PREFIX
)

MEMBERSHIP = pd.DataFrame({
    "row_id": ROW_MAP["row_id"].astype("string"),
    "feature_fingerprint": ROW_MAP["feature_fingerprint"].astype("string"),
    "upstream_fingerprint_group_class": ROW_MAP[
        "fingerprint_group_class"
    ].astype("string"),
    "partition": ROW_MAP["partition"].astype("string"),
    "binary_label": ROW_MAP["binary_label"].astype("int8"),
    "attack_family": ROW_MAP["attack_family_metadata"].astype("string"),
    "source_file": source_file,
    "source_row_number": source_row,
})

allocation_class_map = GROUPS.set_index("feature_fingerprint")[
    "allocation_group_class"
]
MEMBERSHIP["fingerprint_group_class"] = MEMBERSHIP[
    "feature_fingerprint"
].map(allocation_class_map).astype("string")

MEMBERSHIP["reserve_reason_if_applicable"] = MEMBERSHIP[
    "feature_fingerprint"
].map(RESERVE_REASON).astype("string")

# Only attack rows carry attack-family metadata in the common field.
MEMBERSHIP.loc[
    MEMBERSHIP["binary_label"].eq(0), "attack_family"
] = pd.NA

# Dataset-specific metadata enrichment happens in the following cell.
print("Candidate membership rows:", f"{len(MEMBERSHIP):,}")
display(MEMBERSHIP.head(10))


Candidate membership rows: 2,830,743


,row_id,feature_fingerprint,upstream_fingerprint_group_class,partition,binary_label,attack_family,source_file,source_row_number,fingerprint_group_class,reserve_reason_if_applicable
0,CICIDS2017::Monday-WorkingHours.pcap_ISCX.csv:...,h128:f360ff82f6b37201f360ff82f6b37201,unique,development,0,<NA>,Monday-WorkingHours.pcap_ISCX.csv,1,pure benign group,<NA>
1,CICIDS2017::Monday-WorkingHours.pcap_ISCX.csv:...,h128:beafb52511cdd91ebeafb52511cdd91e,same-label duplicate group,benign_early_stop,0,<NA>,Monday-WorkingHours.pcap_ISCX.csv,2,pure benign group,<NA>
2,CICIDS2017::Monday-WorkingHours.pcap_ISCX.csv:...,h128:beafb52511cdd91ebeafb52511cdd91e,same-label duplicate group,benign_early_stop,0,<NA>,Monday-WorkingHours.pcap_ISCX.csv,3,pure benign group,<NA>
3,CICIDS2017::Monday-WorkingHours.pcap_ISCX.csv:...,h128:beafb52511cdd91ebeafb52511cdd91e,same-label duplicate group,benign_early_stop,0,<NA>,Monday-WorkingHours.pcap_ISCX.csv,4,pure benign group,<NA>
4,CICIDS2017::Monday-WorkingHours.pcap_ISCX.csv:...,h128:3dea6ac446942c233dea6ac446942c23,unique,benign_weight_train,0,<NA>,Monday-WorkingHours.pcap_ISCX.csv,5,pure benign group,<NA>
5,CICIDS2017::Monday-WorkingHours.pcap_ISCX.csv:...,h128:13a0db9cc9913c5b13a0db9cc9913c5b,same-label duplicate group,benign_early_stop,0,<NA>,Monday-WorkingHours.pcap_ISCX.csv,6,pure benign group,<NA>
6,CICIDS2017::Monday-WorkingHours.pcap_ISCX.csv:...,h128:13a0db9cc9913c5b13a0db9cc9913c5b,same-label duplicate group,benign_early_stop,0,<NA>,Monday-WorkingHours.pcap_ISCX.csv,7,pure benign group,<NA>
7,CICIDS2017::Monday-WorkingHours.pcap_ISCX.csv:...,h128:13a0db9cc9913c5b13a0db9cc9913c5b,same-label duplicate group,benign_early_stop,0,<NA>,Monday-WorkingHours.pcap_ISCX.csv,8,pure benign group,<NA>
8,CICIDS2017::Monday-WorkingHours.pcap_ISCX.csv:...,h128:fcd2c82706da6e9efcd2c82706da6e9e,unique,benign_threshold_calibration,0,<NA>,Monday-WorkingHours.pcap_ISCX.csv,9,pure benign group,<NA>
9,CICIDS2017::Monday-WorkingHours.pcap_ISCX.csv:...,h128:c4036caa4a99f082c4036caa4a99f082,unique,benign_weight_train,0,<NA>,Monday-WorkingHours.pcap_ISCX.csv,10,pure benign group,<NA>


In [ ]:
# ============================================================
# CICIDS2017 DATASET-SPECIFIC MEMBERSHIP METADATA
# ============================================================
# The authoritative CICIDS attack_family is the frozen supplied attack label for
# attack observations. No family collapse or alternate taxonomy is introduced here.

attack_mask = MEMBERSHIP["binary_label"].eq(1)
if MEMBERSHIP.loc[attack_mask, "attack_family"].isna().any():
    raise RuntimeError("CICIDS attack observations have missing attack-family metadata.")


# Section 11 — Hard partition-integrity gate and membership freeze


In [ ]:
# ============================================================
# 23–25. SECTION 10 MEMBERSHIP FREEZE + SECTION 11 HARD INTEGRITY GATE
# ============================================================
INTEGRITY_CHECKS = []

def add_check(name, passed, hard=True, detail=""):
    passed = bool(passed)
    INTEGRITY_CHECKS.append({
        "check": name,
        "passed": passed,
        "hard": bool(hard),
        "detail": str(detail),
    })
    if hard and not passed:
        record_failure(f"{name}: {detail}")

# Observation completeness and uniqueness.
add_check(
    "membership_row_count_equals_master",
    len(MEMBERSHIP) == MASTER_ROW_COUNT,
    detail=f"membership={len(MEMBERSHIP)}, master={MASTER_ROW_COUNT}",
)
add_check(
    "row_id_non_null",
    not MEMBERSHIP["row_id"].isna().any(),
)
add_check(
    "row_id_unique",
    not MEMBERSHIP["row_id"].duplicated().any(),
)

# Every feature fingerprint must map to exactly one membership state.
fp_partition_nunique = MEMBERSHIP.groupby(
    "feature_fingerprint", sort=False
)["partition"].nunique()
add_check(
    "no_feature_fingerprint_crosses_membership_states",
    int((fp_partition_nunique > 1).sum()) == 0,
    detail=f"crossing_fingerprints={int((fp_partition_nunique > 1).sum())}",
)

# Benign-only purity.
for p in BENIGN_ONLY_PARTITIONS:
    attacks = int(
        MEMBERSHIP.loc[MEMBERSHIP["partition"].eq(p), "binary_label"].sum()
    )
    add_check(
        f"{p}_contains_zero_attacks",
        attacks == 0,
        detail=f"attack_rows={attacks}",
    )

# Mixed-label protection.
mixed_mask = MEMBERSHIP["fingerprint_group_class"].eq(
    "mixed-label/conflicting group"
)
mixed_states = set(
    MEMBERSHIP.loc[mixed_mask, "partition"].dropna().astype(str).unique()
)
add_check(
    "mixed_label_groups_only_in_development_or_final_test",
    mixed_states.issubset(set(MIXED_EVALUATION_PARTITIONS)),
    detail=f"observed_states={sorted(mixed_states)}",
)

# Reserve purity: reserve may contain pure attack only.
reserve = MEMBERSHIP["partition"].eq("attack_reserve_unused")
reserve_benign = int(
    MEMBERSHIP.loc[reserve, "binary_label"].eq(0).sum()
)
reserve_mixed = int(
    MEMBERSHIP.loc[
        reserve,
        "fingerprint_group_class",
    ].eq("mixed-label/conflicting group").sum()
)
add_check(
    "attack_reserve_contains_no_benign",
    reserve_benign == 0,
    detail=f"benign_rows_in_reserve={reserve_benign}",
)
add_check(
    "attack_reserve_contains_no_mixed_label_groups",
    reserve_mixed == 0,
    detail=f"mixed_rows_in_reserve={reserve_mixed}",
)

# Composition by mixed partition.
composition = (
    MEMBERSHIP.groupby(["partition", "binary_label"], sort=False)
    .size()
    .rename("rows")
    .reset_index()
)

def class_count(partition, label):
    x = composition.loc[
        composition["partition"].eq(partition)
        & composition["binary_label"].eq(label),
        "rows",
    ]
    return int(x.iloc[0]) if len(x) else 0

DEV_B = class_count("development", 0)
DEV_A = class_count("development", 1)
TEST_B = class_count("final_test", 0)
TEST_A = class_count("final_test", 1)
RESERVE_A = class_count("attack_reserve_unused", 1)

add_check(
    "development_contains_benign_and_attack",
    DEV_B > 0 and DEV_A > 0,
    detail=f"benign={DEV_B}, attack={DEV_A}",
)
add_check(
    "final_test_contains_benign_and_attack",
    TEST_B > 0 and TEST_A > 0,
    detail=f"benign={TEST_B}, attack={TEST_A}",
)

DEV_PREV = DEV_A / (DEV_A + DEV_B) if (DEV_A + DEV_B) else np.nan
TEST_PREV = TEST_A / (TEST_A + TEST_B) if (TEST_A + TEST_B) else np.nan

add_check(
    "development_strict_benign_majority",
    DEV_B > DEV_A,
    detail=f"benign={DEV_B}, attack={DEV_A}",
)
add_check(
    "final_test_strict_benign_majority",
    TEST_B > TEST_A,
    detail=f"benign={TEST_B}, attack={TEST_A}",
)
add_check(
    "development_attack_prevalence_within_ceiling",
    DEV_PREV <= MAX_MIXED_ATTACK_PREVALENCE,
    detail=f"prevalence={DEV_PREV:.12f}, ceiling={MAX_MIXED_ATTACK_PREVALENCE}",
)
add_check(
    "final_test_attack_prevalence_within_ceiling",
    TEST_PREV <= MAX_MIXED_ATTACK_PREVALENCE,
    detail=f"prevalence={TEST_PREV:.12f}, ceiling={MAX_MIXED_ATTACK_PREVALENCE}",
)

# Benign accounting: every benign observation is in exactly one experimental role.
benign_membership = MEMBERSHIP.loc[MEMBERSHIP["binary_label"].eq(0)]
add_check(
    "all_benign_used_exactly_once",
    len(benign_membership) == B_TOTAL
    and set(benign_membership["partition"].astype(str).unique()).issubset(
        set(EXPERIMENTAL_PARTITIONS)
    ),
    detail=f"accounted_benign={len(benign_membership)}, B={B_TOTAL}",
)

# Attack accounting.
accounted_attack = DEV_A + TEST_A + RESERVE_A
add_check(
    "all_attacks_accounted",
    accounted_attack == A_TOTAL,
    detail=f"accounted_attack={accounted_attack}, A={A_TOTAL}",
)

# Protected metadata / frozen feature policy.
protected_norms = {
    "".join(ch for ch in str(x).casefold() if ch.isalnum())
    for x in [
        "binary_label",
        "attack_family",
        "attack_cat_original",
        "attack_cat_canonical",
        "row_id",
        "feature_fingerprint",
        "source_file",
        "source_row_number",
        "partition",
        "reserve_reason",
    ]
}
model_columns = RAW_FEATURE_POLICY.get("included_model_columns", [])
model_norms = {
    "".join(ch for ch in str(x).casefold() if ch.isalnum())
    for x in model_columns
}
metadata_leaks = sorted(protected_norms & model_norms)
add_check(
    "partition_metadata_not_promoted_to_model_inputs",
    len(metadata_leaks) == 0,
    detail=f"unexpected_model_metadata={metadata_leaks}",
)

# No row omitted or duplicated is already implied by unique row_id + row count.
# Verify reserve reasons are explicit and limited to the frozen vocabulary.
allowed_reserve_reasons = {
    "BENIGN_MAJORITY_CAP",
    "ATOMIC_GROUP_CAPACITY",
    "FAMILY_ALLOCATION_CONSTRAINT",
    "RARE_FAMILY_CAPACITY_CONFLICT",
}
reserve_reasons = set(
    MEMBERSHIP.loc[
        reserve, "reserve_reason_if_applicable"
    ].dropna().astype(str).unique()
)
add_check(
    "reserve_reasons_are_explicit",
    reserve_reasons.issubset(allowed_reserve_reasons),
    detail=f"reserve_reasons={sorted(reserve_reasons)}",
)

INTEGRITY_DF = pd.DataFrame(INTEGRITY_CHECKS)

if BLOCKING_FAILURES:
    FINAL_STATUS = "FAIL"
else:
    FINAL_STATUS = (
        "PASS_WITH_NONBLOCKING_WARNINGS"
        if NONBLOCKING_WARNINGS
        else "PASS"
    )

# Do not freeze an authoritative membership file when hard integrity failed.
PARTITION_MEMBERSHIP_PATH = OUTPUT_DIR / "partition_membership.parquet"
MODEL_ACCESS_PATH = OUTPUT_DIR / "model_development_partition_access.parquet"

CANONICAL_MEMBERSHIP_HASH = None
PARTITION_MANIFEST_FILE_SHA256 = None

if FINAL_STATUS != "FAIL":
    # Deterministic column order.
    membership_columns = [
        "row_id",
        "feature_fingerprint",
        "fingerprint_group_class",
        "upstream_fingerprint_group_class",
        "partition",
        "binary_label",
        "attack_family",
        "source_file",
        "source_row_number",
        "reserve_reason_if_applicable",
    ]
    for extra in ["attack_cat_original", "attack_cat_canonical"]:
        if extra in MEMBERSHIP.columns:
            membership_columns.append(extra)

    MEMBERSHIP = MEMBERSHIP[membership_columns].copy()

    CANONICAL_MEMBERSHIP_HASH = canonical_membership_hash(MEMBERSHIP)
    write_parquet_frozen(PARTITION_MEMBERSHIP_PATH, MEMBERSHIP)
    PARTITION_MANIFEST_FILE_SHA256 = sha256_file(PARTITION_MEMBERSHIP_PATH)

    # Governance-safe convenience view for Notebooks 00–06.
    access_view = MEMBERSHIP[
        ["row_id", "partition", "feature_fingerprint"]
    ].copy()
    write_parquet_frozen(MODEL_ACCESS_PATH, access_view)
    del access_view
else:
    print(
        "Hard integrity failure: authoritative partition_membership.parquet "
        "will NOT be frozen."
    )

display(INTEGRITY_DF)


,check,passed,hard,detail
0,membership_row_count_equals_master,True,True,"membership=2830743, master=2830743"
1,row_id_non_null,True,True,
2,row_id_unique,True,True,
3,no_feature_fingerprint_crosses_membership_states,True,True,crossing_fingerprints=0
4,benign_weight_train_contains_zero_attacks,True,True,attack_rows=0
5,benign_early_stop_contains_zero_attacks,True,True,attack_rows=0
6,benign_threshold_calibration_contains_zero_att...,True,True,attack_rows=0
7,mixed_label_groups_only_in_development_or_fina...,True,True,observed_states=['development']
8,attack_reserve_contains_no_benign,True,True,benign_rows_in_reserve=0
9,attack_reserve_contains_no_mixed_label_groups,True,True,mixed_rows_in_reserve=0


# Allocation, prevalence, and family summaries


In [ ]:
# ============================================================
# 26. ALLOCATION / FAMILY / PREVALENCE SUMMARIES
# ============================================================
PARTITION_SUMMARY_DF = (
    MEMBERSHIP.groupby(["partition", "binary_label"], sort=False)
    .size()
    .rename("rows")
    .reset_index()
)

# Family achieved rows and groups by state.
attack_membership = MEMBERSHIP.loc[
    MEMBERSHIP["binary_label"].eq(1)
].copy()

family_rows_by_partition = (
    attack_membership.groupby(
        ["attack_family", "partition"],
        sort=False,
        dropna=False,
    )
    .size()
    .rename("row_support")
    .reset_index()
)

family_groups_by_partition = (
    attack_membership.groupby(
        ["attack_family", "partition"],
        sort=False,
        dropna=False,
    )["feature_fingerprint"]
    .nunique()
    .rename("group_support")
    .reset_index()
)

family_achieved = family_rows_by_partition.merge(
    family_groups_by_partition,
    on=["attack_family", "partition"],
    how="outer",
)

def fam_value(df, family, partition, column):
    x = df.loc[
        df["attack_family"].astype("string").eq(str(family))
        & df["partition"].astype("string").eq(partition),
        column,
    ]
    return int(x.iloc[0]) if len(x) else 0

family_rows = []
for target in FAMILY_TARGET_ROWS.itertuples(index=False):
    fam = str(target.family)
    dev_rows = fam_value(
        family_achieved, fam, "development", "row_support"
    )
    test_rows = fam_value(
        family_achieved, fam, "final_test", "row_support"
    )
    reserve_rows = fam_value(
        family_achieved, fam, "attack_reserve_unused", "row_support"
    )

    dev_groups = fam_value(
        family_achieved, fam, "development", "group_support"
    )
    test_groups = fam_value(
        family_achieved, fam, "final_test", "group_support"
    )
    reserve_groups = fam_value(
        family_achieved, fam, "attack_reserve_unused", "group_support"
    )

    independent_groups = int(target.independent_fingerprint_group_support)

    if independent_groups == 1:
        if test_groups >= 1:
            status = "RARE_SINGLE_GROUP_FINAL_TEST"
            reason = (
                "one independent fingerprint group; final-test preference satisfied; "
                "later family estimate is descriptive/high-uncertainty and not development-informed"
            )
        elif dev_groups >= 1:
            status = "RARE_SINGLE_GROUP_DEVELOPMENT"
            reason = (
                "one independent fingerprint group; higher-priority constraints prevented "
                "preferred final-test placement"
            )
        else:
            status = "RARE_SINGLE_GROUP_RESERVED"
            reason = (
                "one independent fingerprint group; higher-priority hard capacity "
                "prevented use in mixed partitions"
            )
    elif dev_groups >= 1 and test_groups >= 1:
        status = "REPRESENTED_IN_BOTH"
        reason = "independent family support permitted development and final-test representation"
    elif reserve_rows > 0:
        status = "PARTIAL_OR_RESERVED"
        reason = "atomic/hard-capacity constraints prevented complete target realization"
    else:
        status = "ONE_MIXED_PARTITION_ONLY"
        reason = "group indivisibility/family-target trade-offs prevented both-partition representation"

    family_rows.append({
        "family": fam,
        "total_row_support": int(target.total_row_support),
        "independent_fingerprint_group_support": independent_groups,
        "target_development_rows": int(target.target_development_rows),
        "target_final_test_rows": int(target.target_final_test_rows),
        "target_selected_rows": int(target.target_selected_rows),
        "development_row_support": dev_rows,
        "development_group_support": dev_groups,
        "final_test_row_support": test_rows,
        "final_test_group_support": test_groups,
        "reserve_row_support": reserve_rows,
        "reserve_group_support": reserve_groups,
        "achieved_selected_rows": dev_rows + test_rows,
        "development_target_deviation": (
            dev_rows - int(target.target_development_rows)
        ),
        "final_test_target_deviation": (
            test_rows - int(target.target_final_test_rows)
        ),
        "status": status,
        "reason": reason,
    })

FAMILY_ALLOCATION_DF = pd.DataFrame(family_rows)

RARE_FAMILY_EXCEPTIONS_DF = FAMILY_ALLOCATION_DF.loc[
    (FAMILY_ALLOCATION_DF["independent_fingerprint_group_support"] <= 1)
    | ~FAMILY_ALLOCATION_DF["status"].eq("REPRESENTED_IN_BOTH")
].copy()

DEV_RATIO = DEV_A / DEV_B if DEV_B else np.inf
TEST_RATIO = TEST_A / TEST_B if TEST_B else np.inf
PREVALENCE_DF = pd.DataFrame([{
    "development_benign": DEV_B,
    "development_attack": DEV_A,
    "development_attack_prevalence": DEV_PREV,
    "development_attack_to_benign_ratio": DEV_RATIO,
    "final_test_benign": TEST_B,
    "final_test_attack": TEST_A,
    "final_test_attack_prevalence": TEST_PREV,
    "final_test_attack_to_benign_ratio": TEST_RATIO,
    "absolute_prevalence_difference": abs(DEV_PREV - TEST_PREV),
    "allocator_objective_note": (
        "minimized subject to higher-priority atomicity, purity, benign-majority, "
        "benign-accounting, attack-utilization, and rare-family constraints"
    ),
}])

attacks_used = DEV_A + TEST_A
attacks_reserved = RESERVE_A

ATTACK_ALLOCATION_MANIFEST = {
    "dataset": DATASET_NAME,
    "protocol_version": PROTOCOL_VERSION,
    "created_at_utc": PARTITION_CREATED_UTC,
    "partition_policy_hash": PARTITION_POLICY_HASH,
    "PARTITION_SEED": int(PARTITION_SEED),
    "B_total": B_TOTAL,
    "A_total": A_TOTAL,
    "benign_achieved_counts": BENIGN_ACHIEVED,
    "mixed_label_group_assignments": MIXED_ASSIGNMENT_ROWS,
    "A_dev_cap": A_DEV_CAP,
    "A_test_cap": A_TEST_CAP,
    "A_capacity_total": A_CAPACITY_TOTAL,
    "A_use_target": A_USE_TARGET,
    "A_dev_target": A_DEV_TARGET,
    "A_test_target": A_TEST_TARGET,
    "A_dev_achieved": DEV_A,
    "A_test_achieved": TEST_A,
    "attacks_used": attacks_used,
    "attacks_reserved": attacks_reserved,
    "fraction_attacks_used": attacks_used / A_TOTAL if A_TOTAL else 0.0,
    "fraction_attacks_reserved": attacks_reserved / A_TOTAL if A_TOTAL else 0.0,
    "reserve_reason_counts_by_group": dict(Counter(RESERVE_REASON.values())),
}

display(PARTITION_SUMMARY_DF)
display(PREVALENCE_DF)
display(FAMILY_ALLOCATION_DF)


,partition,binary_label,rows
0,development,0,227309
1,benign_early_stop,0,227309
2,benign_weight_train,0,1136548
3,benign_threshold_calibration,0,340964
4,final_test,0,340967
5,final_test,1,327595
6,development,1,218394
7,attack_reserve_unused,1,11657


,development_benign,development_attack,development_attack_prevalence,development_attack_to_benign_ratio,final_test_benign,final_test_attack,final_test_attack_prevalence,final_test_attack_to_benign_ratio,absolute_prevalence_difference,allocator_objective_note
0,227309,218394,0.489999,0.96078,340967,327595,0.489999,0.960782,4.861298e-07,minimized subject to higher-priority atomicity...


,family,total_row_support,independent_fingerprint_group_support,target_development_rows,target_final_test_rows,target_selected_rows,development_row_support,development_group_support,final_test_row_support,final_test_group_support,reserve_row_support,reserve_group_support,achieved_selected_rows,development_target_deviation,final_test_target_deviation,status,reason
0,Bot,1966,1953,770,1155,1925,770,770,1155,1142,41,41,1925,0,0,REPRESENTED_IN_BOTH,independent family support permitted developme...
1,DDoS,128027,128016,50140,75211,125351,50140,50136,75211,75204,2676,2676,125351,0,0,REPRESENTED_IN_BOTH,independent family support permitted developme...
2,DoS GoldenEye,10293,10286,4031,6047,10078,4031,4028,6047,6043,215,215,10078,0,0,REPRESENTED_IN_BOTH,independent family support permitted developme...
3,DoS Hulk,231073,172849,90496,135746,226242,90496,68448,135746,99570,4831,4831,226242,0,0,REPRESENTED_IN_BOTH,independent family support permitted developme...
4,DoS Slowhttptest,5499,5228,2154,3231,5385,2154,1972,3231,3142,114,114,5385,0,0,REPRESENTED_IN_BOTH,independent family support permitted developme...
5,DoS slowloris,5796,5385,2270,3405,5675,2270,2100,3405,3164,121,121,5675,0,0,REPRESENTED_IN_BOTH,independent family support permitted developme...
6,FTP-Patator,7938,5933,3109,4663,7772,3109,1980,4663,3787,166,166,7772,0,0,REPRESENTED_IN_BOTH,independent family support permitted developme...
7,Heartbleed,11,11,4,7,11,4,4,7,7,0,0,11,0,0,REPRESENTED_IN_BOTH,independent family support permitted developme...
8,Infiltration,36,36,14,21,35,14,14,21,21,1,1,35,0,0,REPRESENTED_IN_BOTH,independent family support permitted developme...
9,PortScan,158930,90819,62243,93365,155608,62243,34735,93365,52762,3322,3322,155608,0,0,REPRESENTED_IN_BOTH,independent family support permitted developme...


# Preprocessing & Representation / Notebook 00 handoff


In [ ]:
# ============================================================
# 24. PREPROCESSING HANDOFF MANIFEST — SPECIFICATION ONLY
# ============================================================
if FINAL_STATUS == "FAIL":
    PREPROCESSING_HANDOFF = {
        "dataset": DATASET_NAME,
        "status": "NOT_AUTHORIZED_DUE_TO_PARTITION_FAILURE",
        "next_authorized_preprocessing_rule": "NONE",
    }
else:
    metadata_columns = RAW_FEATURE_POLICY.get("metadata_columns", [])

    PREPROCESSING_HANDOFF = {
        "dataset": DATASET_NAME,
        "protocol_version": PROTOCOL_VERSION,
        "created_at_utc": PARTITION_CREATED_UTC,
        "master_dataset_sha256": MASTER_SHA256,
        "partition_manifest_canonical_hash": CANONICAL_MEMBERSHIP_HASH,
        "partition_manifest_file_sha256": PARTITION_MANIFEST_FILE_SHA256,
        "partition_policy_hash": PARTITION_POLICY_HASH,
        "raw_feature_policy_path": str(RAW_FEATURE_POLICY_PATH),
        "raw_feature_policy_hash": RAW_FEATURE_POLICY_HASH,
        "numerical_model_columns": RAW_FEATURE_POLICY.get(
            "numerical_model_columns", []
        ),
        "categorical_model_columns": RAW_FEATURE_POLICY.get(
            "categorical_model_columns", []
        ),
        "model_input_column_order": RAW_FEATURE_POLICY.get(
            "column_order", []
        ),
        "metadata_columns": metadata_columns,
        "label_columns": DATASET_LABEL_COLUMNS,
        "attack_family_columns": DATASET_ATTACK_FAMILY_COLUMNS,
        "row_identity_fields": [
            "row_id",
            "source_file",
            "source_row_number",
        ],
        "partition_names": EXPERIMENTAL_PARTITIONS,
        "optional_governance_state": "attack_reserve_unused",
        "partition_row_counts": {
            str(k): int(v)
            for k, v in MEMBERSHIP["partition"].value_counts().to_dict().items()
        },
        "benign_only_partition_names": BENIGN_ONLY_PARTITIONS,
        "mixed_evaluation_partition_names": MIXED_EVALUATION_PARTITIONS,
        "final_test_sealing_status": "FROZEN_AND_EVALUATION_SEALED",
        "final_test_transformation_rule": (
            "Final-test transformation is allowed; final-test influence is forbidden."
        ),
        "fit_rule": {
            "learned_preprocessing_fit_partition": "benign_weight_train",
            "fit_on_any_other_partition": False,
        },
        "transform_rule": {
            "apply_frozen_fitted_pipeline_to": EXPERIMENTAL_PARTITIONS,
            "includes_final_test": True,
        },
        "future_learned_components_examples": [
            "numerical missing-value imputation",
            "standardization",
            "categorical vocabulary / unknown handling where applicable",
            "one-hot encoding where applicable",
        ],
        "final_test_permitted_before_notebook_07": [
            "load frozen final-test membership",
            "extract frozen raw model-input columns",
            "apply preprocessor.transform using objects fitted only on benign_weight_train",
            "apply frozen unknown-category handling",
            "verify transformed feature names/order/dimensionality",
            "record transformed-file hash",
            "record row count",
            "record schema compatibility",
        ],
        "final_test_prohibited_before_notebook_07": [
            "preprocessor.fit(final_test)",
            "preprocessor.fit_transform(final_test)",
            "derive imputation statistics from final_test",
            "derive scaler statistics from final_test",
            "derive categorical vocabulary from final_test",
            "derive rare-category thresholds from final_test",
            "derive clipping limits from final_test",
            "change preprocessing/model policy because of final-test values",
            "inspect final-test reconstruction/anomaly-score distributions for tuning",
            "compute or interpret final-test model-performance metrics",
            "inspect family-level detection outcomes",
            "optimize threshold/model/configuration using final-test outcomes",
        ],
        "structural_compatibility_nuance": (
            "Unknown-category counts, transformed out-of-range values, missingness "
            "patterns, or similar final-test structural phenomena may reveal distributional "
            "information. Mechanical checks may be computed if required, preferably stored "
            "without interpretation, and must never drive preprocessing/model-design changes."
        ),
        "next_authorized_preprocessing_rule": (
            "Notebook 00 may FIT learned preprocessing on benign_weight_train only "
            "and TRANSFORM all five frozen experimental partitions using the same "
            "frozen fitted objects."
        ),
    }

PREPROCESSING_HANDOFF_PATH = OUTPUT_DIR / "preprocessing_handoff_manifest.json"
write_json_frozen(PREPROCESSING_HANDOFF_PATH, PREPROCESSING_HANDOFF)


('WRITTEN', 'a9a07aa86ef7771a4d424de41ae96f952417ab185f10d55dff0e817a5eb7b52c')

# Integrity report and artifact inventory


In [ ]:
# ============================================================
# 26–27. WRITE SUMMARIES, INTEGRITY REPORT, AND ARTIFACT INVENTORY
# ============================================================
BENIGN_ALLOCATION_PATH = OUTPUT_DIR / "benign_allocation_table.csv"
ATTACK_ALLOCATION_PATH = OUTPUT_DIR / "attack_allocation_manifest.json"
FAMILY_ALLOCATION_PATH = OUTPUT_DIR / "family_allocation_table.csv"
PREVALENCE_PATH = OUTPUT_DIR / "prevalence_comparison_table.csv"
RARE_EXCEPTIONS_PATH = OUTPUT_DIR / "rare_family_exceptions.csv"
PARTITION_SUMMARY_PATH = OUTPUT_DIR / "partition_manifest_summary.csv"
INTEGRITY_JSON_PATH = OUTPUT_DIR / "partition_integrity_report.json"
INTEGRITY_MD_PATH = OUTPUT_DIR / "partition_integrity_report.md"

write_csv_frozen(BENIGN_ALLOCATION_PATH, BENIGN_ALLOCATION_DF)
write_json_frozen(ATTACK_ALLOCATION_PATH, ATTACK_ALLOCATION_MANIFEST)
write_csv_frozen(FAMILY_ALLOCATION_PATH, FAMILY_ALLOCATION_DF)
write_csv_frozen(PREVALENCE_PATH, PREVALENCE_DF)
write_csv_frozen(RARE_EXCEPTIONS_PATH, RARE_FAMILY_EXCEPTIONS_DF)
write_csv_frozen(PARTITION_SUMMARY_PATH, PARTITION_SUMMARY_DF)

INTEGRITY_REPORT = {
    "dataset": DATASET_NAME,
    "protocol_version": PROTOCOL_VERSION,
    "created_at_utc": PARTITION_CREATED_UTC,
    "upstream_hashes": UPSTREAM_HASHES,
    "PARTITION_SEED": int(PARTITION_SEED),
    "allocation_algorithm_version": ALLOCATION_ALGORITHM_VERSION,
    "partition_policy_hash": PARTITION_POLICY_HASH,
    "partition_manifest_canonical_hash": CANONICAL_MEMBERSHIP_HASH,
    "partition_manifest_file_sha256": PARTITION_MANIFEST_FILE_SHA256,
    "protocol_requirements": {
        "allocation_unit": "feature_fingerprint group",
        "no_benign_reserve": True,
        "strict_benign_majority": True,
        "MAX_MIXED_ATTACK_PREVALENCE": MAX_MIXED_ATTACK_PREVALENCE,
        "fit_preprocessing_later_on": "benign_weight_train only",
        "final_test_rule": "transformation permitted; influence forbidden",
    },
    "observed_allocation": {
        "B_total": B_TOTAL,
        "A_total": A_TOTAL,
        "benign_achieved_counts": BENIGN_ACHIEVED,
        "development_benign": DEV_B,
        "development_attack": DEV_A,
        "development_attack_prevalence": DEV_PREV,
        "final_test_benign": TEST_B,
        "final_test_attack": TEST_A,
        "final_test_attack_prevalence": TEST_PREV,
        "absolute_prevalence_difference": abs(DEV_PREV - TEST_PREV),
        "attack_reserve_rows": RESERVE_A,
    },
    "deviations_from_nominal_targets": BENIGN_ALLOCATION_DF.to_dict(
        orient="records"
    ),
    "warnings": NONBLOCKING_WARNINGS,
    "hard_validation_results": INTEGRITY_DF.to_dict(orient="records"),
    "blocking_failures": BLOCKING_FAILURES,
    "final_status": FINAL_STATUS,
    "final_test_status": (
        "FROZEN_AND_EVALUATION_SEALED"
        if FINAL_STATUS != "FAIL"
        else "NOT_AUTHORIZED"
    ),
    "downstream_handoff": PREPROCESSING_HANDOFF,
    "next_authorized_stage": (
        "Notebook 00: fit preprocessing on benign_weight_train only; transform all "
        "frozen experimental partitions; freeze transformed representation and "
        "DAE corruption groups."
        if FINAL_STATUS != "FAIL"
        else "NONE"
    ),
}

write_json_frozen(INTEGRITY_JSON_PATH, INTEGRITY_REPORT)

md_lines = [
    f"# Partition Integrity Report — {DATASET_NAME}",
    "",
    "## Protocol requirements",
    "",
    "- Allocation unit: immutable `feature_fingerprint` group.",
    "- No benign reserve.",
    f"- Strict benign majority; attack-prevalence ceiling = {MAX_MIXED_ATTACK_PREVALENCE:.2f}.",
    "- Learned preprocessing later fits on `benign_weight_train` only.",
    "- Final-test transformation later is permitted; final-test influence is forbidden.",
    "",
    "## Observed allocation",
    "",
    f"- Total benign: {B_TOTAL:,}",
    f"- Total attack: {A_TOTAL:,}",
    f"- Development: benign={DEV_B:,}, attack={DEV_A:,}, prevalence={DEV_PREV:.8f}",
    f"- Final test: benign={TEST_B:,}, attack={TEST_A:,}, prevalence={TEST_PREV:.8f}",
    f"- Absolute prevalence difference: {abs(DEV_PREV - TEST_PREV):.8f}",
    f"- Attack reserve: {RESERVE_A:,}",
    "",
    "## Deviations from nominal benign targets",
    "",
    BENIGN_ALLOCATION_DF.to_markdown(index=False),
    "",
    "## Hard validation results",
    "",
    INTEGRITY_DF.to_markdown(index=False),
    "",
    "## Warnings",
]
if NONBLOCKING_WARNINGS:
    md_lines.extend([f"- {x}" for x in NONBLOCKING_WARNINGS])
else:
    md_lines.append("- None")

md_lines.extend([
    "",
    "## Blocking failures",
])
if BLOCKING_FAILURES:
    md_lines.extend([f"- {x}" for x in BLOCKING_FAILURES])
else:
    md_lines.append("- None")

md_lines.extend([
    "",
    "## Final status",
    "",
    f"**{FINAL_STATUS}**",
    "",
    "## Downstream handoff",
    "",
    (
        "Notebook 00 is authorized to fit learned preprocessing on "
        "`benign_weight_train` only and then transform all five frozen experimental "
        "partitions, including `final_test`, with the same frozen fitted pipeline."
        if FINAL_STATUS != "FAIL"
        else "No downstream stage is authorized."
    ),
    "",
    (
        "**Final test is evaluation-sealed:** mechanical transformation later is "
        "allowed; its values, labels, scores, metrics, family behavior, or structural "
        "diagnostics must not influence preprocessing/model development before Notebook 07."
    ),
])

write_text_frozen(INTEGRITY_MD_PATH, "\n".join(md_lines) + "\n")

# Attack reserve row-level manifest only if non-empty and integrity passed.
ATTACK_RESERVE_PATH = OUTPUT_DIR / "attack_reserve_manifest.parquet"
if FINAL_STATUS != "FAIL" and RESERVE_A > 0:
    reserve_df = MEMBERSHIP.loc[
        MEMBERSHIP["partition"].eq("attack_reserve_unused")
    ].copy()
    write_parquet_frozen(ATTACK_RESERVE_PATH, reserve_df)
    del reserve_df

# Artifact inventory.
artifact_specs = [
    (PARTITION_POLICY_PATH, "Frozen Sections 8–11 partition policy", "8"),
    (BENIGN_ALLOCATION_PATH, "Benign target/achieved allocation", "8"),
    (ATTACK_ALLOCATION_PATH, "Attack allocation provenance", "9"),
    (FAMILY_ALLOCATION_PATH, "Family target/achieved allocation", "9"),
    (PREVALENCE_PATH, "Development/final prevalence comparison", "9"),
    (RARE_EXCEPTIONS_PATH, "Rare/family allocation exceptions", "9"),
    (PARTITION_SUMMARY_PATH, "Partition/class row-count summary", "10"),
    (PREPROCESSING_HANDOFF_PATH, "Notebook 00 preprocessing handoff specification", "10"),
    (INTEGRITY_JSON_PATH, "Machine-readable hard integrity report", "11"),
    (INTEGRITY_MD_PATH, "Human-readable hard integrity report", "11"),
]

if FINAL_STATUS != "FAIL":
    artifact_specs.extend([
        (PARTITION_MEMBERSHIP_PATH, "Authoritative immutable row membership", "10"),
        (MODEL_ACCESS_PATH, "Governance-safe model-development access view", "10"),
    ])
    if RESERVE_A > 0:
        artifact_specs.append(
            (ATTACK_RESERVE_PATH, "Row-level unused pure-attack reserve", "10")
        )

ARTIFACT_RECORDS = []
for path, purpose, section in artifact_specs:
    if Path(path).exists():
        register_artifact(path, purpose, section)

ARTIFACT_INVENTORY_PATH = OUTPUT_DIR / "partition_artifact_inventory.csv"
write_csv_frozen(
    ARTIFACT_INVENTORY_PATH,
    pd.DataFrame(ARTIFACT_RECORDS).drop_duplicates(
        subset=["artifact_path"], keep="last"
    ),
)

display(pd.DataFrame(ARTIFACT_RECORDS))


,artifact_name,artifact_path,artifact_type,file_size,sha256,purpose,section
0,partition_policy_manifest.json,/content/exp1v2_google_drive/MyDrive/EXPERIMEN...,json,3260,13729814b572b27f6b909562ebb60e2aaeb63cac436dd2...,Frozen Sections 8–11 partition policy,8
1,benign_allocation_table.csv,/content/exp1v2_google_drive/MyDrive/EXPERIMEN...,csv,628,93cb098eee9795787b22f92017d01d7a864ee7b593ee38...,Benign target/achieved allocation,8
2,attack_allocation_manifest.json,/content/exp1v2_google_drive/MyDrive/EXPERIMEN...,json,243275,39f732fe8efeea8cecd76d4049e5a61f2928ed4e71e20f...,Attack allocation provenance,9
3,family_allocation_table.csv,/content/exp1v2_google_drive/MyDrive/EXPERIMEN...,csv,2761,e427f95088bd0636f5a087d7a377ba07f8b10ab3dce487...,Family target/achieved allocation,9
4,prevalence_comparison_table.csv,/content/exp1v2_google_drive/MyDrive/EXPERIMEN...,csv,527,a372dfb22a105041ac117a508d8066b35013055123e9f7...,Development/final prevalence comparison,9
5,rare_family_exceptions.csv,/content/exp1v2_google_drive/MyDrive/EXPERIMEN...,csv,365,7fddb28f3470e8fd27922aae0b82b6ff6b910374ee89bb...,Rare/family allocation exceptions,9
6,partition_manifest_summary.csv,/content/exp1v2_google_drive/MyDrive/EXPERIMEN...,csv,235,db3bd1811353be40fe94fd83f9b6421d460d571c546483...,Partition/class row-count summary,10
7,preprocessing_handoff_manifest.json,/content/exp1v2_google_drive/MyDrive/EXPERIMEN...,json,8178,a9a07aa86ef7771a4d424de41ae96f952417ab185f10d5...,Notebook 00 preprocessing handoff specification,10
8,partition_integrity_report.json,/content/exp1v2_google_drive/MyDrive/EXPERIMEN...,json,16190,8ec92e29095ae2421d75030f7d0c5f6b6c44ecad956a87...,Machine-readable hard integrity report,11
9,partition_integrity_report.md,/content/exp1v2_google_drive/MyDrive/EXPERIMEN...,md,5524,504b79027847f11b56a02f612d350112743979844a4065...,Human-readable hard integrity report,11


# Final readiness and next-stage authorization


In [ ]:
# ============================================================
# 28–29. FINAL READINESS RESULT AND NEXT AUTHORIZED STAGE
# ============================================================
NEXT_AUTHORIZED_STAGE = (
    "Notebook 00: fit preprocessing on benign_weight_train only; transform all "
    "frozen experimental partitions; freeze transformed representation and "
    "DAE corruption groups."
    if FINAL_STATUS != "FAIL"
    else "NONE"
)

print("=" * 84)
print(f"Dataset: {DATASET_NAME}")
print("Sections implemented: 8–11")
print()
print(f"Master dataset SHA-256: {MASTER_SHA256}")
print(f"Global protocol hash: {GLOBAL_PROTOCOL_HASH}")
print(f"Fingerprint manifest hash: {FINGERPRINT_MANIFEST_FILE_SHA256}")
print()
print(f"PARTITION_SEED: {PARTITION_SEED}")
print(f"Partition policy hash: {PARTITION_POLICY_HASH}")
print(f"Canonical membership hash: {CANONICAL_MEMBERSHIP_HASH}")
print()
print(f"Total benign observations: {B_TOTAL:,}")
print(f"Total attack observations: {A_TOTAL:,}")
print()
print(
    "benign_weight_train:",
    f"{int((MEMBERSHIP['partition'] == 'benign_weight_train').sum()):,}",
)
print(
    "benign_early_stop:",
    f"{int((MEMBERSHIP['partition'] == 'benign_early_stop').sum()):,}",
)
print(
    "benign_threshold_calibration:",
    f"{int((MEMBERSHIP['partition'] == 'benign_threshold_calibration').sum()):,}",
)
print(f"development benign: {DEV_B:,}")
print(f"development attack: {DEV_A:,}")
print(f"development attack prevalence: {DEV_PREV:.10f}")
print()
print(f"final_test benign: {TEST_B:,}")
print(f"final_test attack: {TEST_A:,}")
print(f"final_test attack prevalence: {TEST_PREV:.10f}")
print()
print(
    "absolute development/final prevalence difference:",
    f"{abs(DEV_PREV - TEST_PREV):.10f}",
)
print()
print(f"attacks used: {DEV_A + TEST_A:,}")
print(f"attacks reserved: {RESERVE_A:,}")
print()
print(
    "fingerprints crossing membership states:",
    int((fp_partition_nunique > 1).sum()),
)
print(
    "attacks in benign-only roles:",
    sum(
        int(
            MEMBERSHIP.loc[
                MEMBERSHIP["partition"].eq(p), "binary_label"
            ].sum()
        )
        for p in BENIGN_ONLY_PARTITIONS
    ),
)
print()
print(f"Blocking failures: {len(BLOCKING_FAILURES)}")
print(f"Nonblocking warnings: {len(NONBLOCKING_WARNINGS)}")
print()
print(f"PARTITION INTEGRITY: {FINAL_STATUS}")
print()
print(
    "Final-test status:",
    (
        "FROZEN AND EVALUATION-SEALED"
        if FINAL_STATUS != "FAIL"
        else "NOT AUTHORIZED"
    ),
)
print()
print("Future preprocessing rule:")
print("FIT benign_weight_train only.")
print(
    "TRANSFORM all five experimental partitions using the frozen fitted pipeline."
)
print(
    "final_test may be mechanically transformed by the benign_weight_train-fitted "
    "frozen preprocessor, but final_test must not influence fitting, configuration, "
    "tuning, thresholding, model selection, or interpretation before Notebook 07."
)
print()
print("Next authorized stage:")
print(NEXT_AUTHORIZED_STAGE)
print("=" * 84)

if FINAL_STATUS == "FAIL":
    raise RuntimeError(
        "Partition integrity FAILED. No preprocessing/modeling stage is authorized."
    )


Dataset: CICIDS2017
Sections implemented: 8–11

Master dataset SHA-256: b68a108f6fe44082a051e2ab81a4bd8859cb3ccd417ff60623871155d51cabd3
Global protocol hash: 55713fd82c01168e0a906d282644b1c8dff88632dcb82512c5f62c637e35ff1f
Fingerprint manifest hash: 1bc46e61bd3c74df45cd21cf280b773693a66898f3a53b744aeb25fd4da00ecf

PARTITION_SEED: 20260914
Partition policy hash: be62dd88cedd77903c19f64c3bab1d5fac3508820d67906f5f17381eddcde66a
Canonical membership hash: 724eeca41efa11a8ff58cba6efd172d851e9ad43870eafa6d0e01956dd8e6e3e

Total benign observations: 2,273,097
Total attack observations: 557,646

benign_weight_train: 1,136,548
benign_early_stop: 227,309
benign_threshold_calibration: 340,964
development benign: 227,309
development attack: 218,394
development attack prevalence: 0.4899989455

final_test benign: 340,967
final_test attack: 327,595
final_test attack prevalence: 0.4899994316

absolute development/final prevalence difference: 0.0000004861

attacks used: 545,989
attacks reserved: 11,65